#### 图像分类
- 最近邻
- 线性分类

计算机如何认识图像？

图像在计算机中表示为多维张量，具有 RGB 三个通道，每个通道的数值范围在 [0, 255] 之间。

传统方法通过边缘检测器寻找角点等特征，但这类方法泛化能力有限。

最近邻效果并不理想  
像素级比较对微小位移、光照变化非常敏感，且测试阶段计算量极大。

数据驱动方法  
1. 收集数据集和标签  
2. 用机器学习算法训练分类器  
3. 在新图像上测试分类器  

其中，训练集通常表示为：

$$
\{(x_i, y_i)\}_{i=1}^N
$$

其中 $x_i$ 表示第 $i$ 张输入图像，$y_i$ 表示其对应的真实标签，$N$ 为样本总数。

模型的目标是学习一个映射函数：

$$
f: X \to Y
$$

使得该函数能够将输入图像 $x$ 正确映射到其类别标签 $y$。

In [ ]:
import numpy as np
from typing import Any

def train(images: np.ndarray, labels: np.ndarray) -> Any:
    """
    Train a machine learning model.
    images: training images, shape (N, D)
    labels: training labels, shape (N,)
    Returns the trained model.
    """
    # Machine learning model training logic
    pass

def predict(model: Any, test_images: np.ndarray) -> np.ndarray:
    """
    Use the trained model to predict labels for test_images.
    model: trained model
    test_images: test images, shape (M, D)
    Returns predicted labels, shape (M,)
    """
    # Prediction logic
    pass

##### 最近邻分类器Nearest Neighbor

Distance Magic

距离计算是最近邻分类器的核心，用于衡量图片像素之间的相似度。

**L1距离**

L1距离又称曼哈顿距离，其几何特性是只能沿着坐标轴横着或竖着移动。

$$
d_1(I_1, I_2) = \sum_p |I_1^p - I_2^p|
$$

其中 $I_1$ 和 $I_2$ 代表两张图像，$p $为像素索引。

L1距离与L2距离的直观对比如下图所示：

![L1与L2距离对比图](../img/L1L2.png)

In [ ]:
import numpy as np
''' 
一个基于L1距离的最近邻图像分类器
它在训练阶段只是“死记硬背”所有图片和标签
在预测阶段则计算新图片与所有训练图片的像素差
找到最相似的那张图，直接把它的标签作为预测结果
'''

'''
输入：

X（图片数据）：是一个二维矩阵（Numpy Array），形状为 (N, D)。

N：图片的数量。比如训练集有50000张图。

D：每张图展平后的像素总数。32×32×3（RGB三通道） = 3072。

所以 X 就是一个 (50000, 3072) 的大矩阵，每一行代表一张被拉直成一条线的图片。

y（真实标签）：是一个一维数组，形状为 (N,)。

比如 y = [0, 5, 3, 2, ...]，里面的数字是 0 到 9，对应 10 个类别。

测试集输入：predict 里的 X 形状是 (M, D)，M 是测试图片数量（比如10000），D 依然是 3072。

'''

'''
输出：

X[i, :] 取出了第 i 张测试图片（一个长度为 D 的一维向量）。

self.Xtr - X[i, :] 利用了 NumPy 的广播机制，让所有训练图片（50000行）都减去这张测试图片。

np.abs(...) 取绝对值。

np.sum(..., axis=1) 把每行加起来，得到 50000 个距离值（L1距离）。

distances 是一个长度为 50000 的数组，代表这张测试图与所有训练图的像素差之和。

找出 distances 中最小值的索引，也就是“最像”的那张训练图的下标。

把最像的那张训练图的标签，直接作为第 i 张测试图的预测结果。

'''
class NearestNeighbor:
    def __init__(self):
        pass

    def train(self, X, y):
        # X is N x D where each row is an example.
        # Y is 1-dimensional of size N.
        # The nearest neighbor classifier simply remembers all the training data.
        self.Xtr = X
        self.ytr = y

    def predict(self, X):
        # X is N x D where each row is an example we wish to predict label for.
        num_test = X.shape[0]

        # Let's make sure that the output type matches the input type.
        Ypred = np.zeros(num_test, dtype=self.ytr.dtype)

        # Loop over all test rows.
        for i in range(num_test):
            # Find the nearest training image to the i'th test image
            # using the L1 distance (sum of absolute value differences).
            distances = np.sum(np.abs(self.Xtr - X[i, :]), axis=1)

            min_index = np.argmin(distances)  # Get the index with smallest distance.
            Ypred[i] = self.ytr[min_index]    # Predict the label of the nearest example.

        return Ypred


**L2距离**

L2距离又称欧氏距离，度量的是两点之间的直线距离。

数学公式如下：

$$
d_2(I_1, I_2) = \sqrt{\sum_p (I_1^p - I_2^p)^2}
$$

其中 $I_1$ 和 $I_2$ 代表两张图像，$p$ 为像素索引。

L1与L2的对比可参考前文的图示。

---

##### K-Nearest Neighbors

K近邻分类器是最近邻的推广。K=1时即为最近邻分类器，预测时取最近的1个训练样本的标签作为结果。K>1时，则选取距离最近的K个训练样本，通过投票决定最终预测类别。

[K-Nearst Neighbors 演示](http:vision.stanford.edu/teaching/cs231n-demos/knn/)

##### 超参数

K近邻算法中的K值和距离函数都是典型的超参数。超参数需要人为设置，不能由算法从数据中自动学习得到。

**K值的选择**

K值过小，模型对噪声敏感，容易过拟合。K=1时在训练集上永远能达到100%准确率，但这不代表模型泛化能力强。

K值过大，模型过于平滑，可能欠拟合。

**超参数选取策略**

常见的超参数选取策略有以下几种。

Idea #1：选择在训练集上表现最好的超参数。这是错误的，因为K=1永远在训练集上完美。

Idea #2：选择在测试集上表现最好的超参数。这也是错误的，这样会导致算法对测试集过拟合，实际部署时性能会远低于预期。

Idea #3：将数据分为训练集、验证集，在验证集上选择超参数，最后在测试集上评估。这是正确的做法。

Idea #4：交叉验证。将训练集分成若干份，轮流将其中一份作为验证集，其余作为训练集，最后取平均结果。适用于数据集较小的情况。

---

##### 数据集划分

**3. 训练集、验证集、测试集**

将可用数据划分为三部分：

- **训练集**：用于训练模型参数。
- **验证集**：用于调整超参数，选择最优配置。
- **测试集**：只在最后使用一次，用于评估最终模型的泛化性能。

以CIFAR-10为例，可以用49000张作为训练集，1000张作为验证集。

**Idea #3的具体流程**

在验证集上尝试不同的超参数，记录每个超参数对应的准确率，选择验证集上表现最好的超参数。然后用这个超参数在全部训练数据上重新训练，最后在测试集上跑一次，报告结果。

**4. 交叉验证**

当训练数据较少时，验证集数量也会很少，此时可以使用交叉验证。

将训练集平均分成 $k$ 份（通常k=3、5、10），每次用其中 $k-1$ 份训练，剩下1份验证，循环 $k$ 次，最后取 $k$ 次验证结果的平均值作为该超参数的性能估计。

这样做的好处是减少了验证集划分带来的噪声，得到更稳定的超参数选择。缺点是计算成本成倍增加。

如果训练数据充足，通常优先使用单次验证集划分，因为交叉验证计算开销较大。

##### 距离度量的局限性

K近邻使用像素级距离进行图像分类，实际效果并不理想。

原因有二：

第一，像素距离对微小的位移、旋转、光照变化非常敏感。同一物体经过平移或旋转后，像素值差异可能很大，导致被判定为不同类别。

第二，高维空间中的距离度量会失去区分度，这种现象被称为维数灾难。随着维度增加，所有点之间的距离趋于接近，距离度量不再具有信息量。

因此，K近邻搭配像素距离在图像分类中几乎不被实际使用。

---

##### 线性分类器 Linear Classifier

**映射**

线性分类器是一个把输入映射到输出的函数 $f(x, W)$，其中 $W$ 是权重矩阵。

输入图像大小为 $32 \times 32 \times 3$，展平后得到长度为 3072 的向量 $x$。函数 $f(x, W)$ 将其映射为 10 个类别的分数。

线性模型公式如下：

$$
f(x, W) = W x + b
$$

一张图像会对每个类别都输出一个分数。

线性分类器是神经网络的基础模块。

![f(x,W)](../img/f(x,W).png)

**损失函数与最大似然估计**

损失函数用于衡量预测分数与真实分数之间的差异。

基于最大似然估计，计算正确类别的概率，取对数，再取负值，就得到了损失。

**Softmax 公式**

Softmax 分类器把原始分数转换为概率分布。它本质上就是多分类逻辑回归。

Softmax 函数将分数 $s$ 映射为合为1概率：

$$
P(y = k | x) = \frac{e^{s_k}}{\sum_j e^{s_j}}
$$

负值概率趋于0。

对应的损失函数即交叉熵损失：

$$
L_i = -\log P(y_i | x_i)
$$

它衡量的是模型预测分布与真实分布之间的差异。


---

**图像识别时计算机会遇到很多挑战。**

![challenges](../img/COR.png)

#### 正则化和优化


##### 正则化 Regularization

核心思路：在训练集上表现稍差，但在未见过的数据上表现更好。倾向于选择拟合度稍低但更简单的模型。

通常不对偏置项进行正则化，因为偏置不控制特征的方向。

完整的损失函数由数据损失和正则化项组成：

$$
L = \frac{1}{N} \sum_i L_i + \lambda R(W)
$$

其中 $\lambda$ 是正则化强度，$R(W)$ 是正则化项。

**L2正则化**

对权重平方进行惩罚。对极小值的惩罚更小，倾向于让权重分散。

$$
R(W) = \sum_k \sum_l W_{k,l}^2
$$

**L1正则化**

对权重绝对值进行惩罚。倾向于产生稀疏的权重矩阵。

$$
R(W) = \sum_k \sum_l |W_{k,l}|
$$

**L1 + L2**

结合两者，又称 Elastic Net。

![Regularization](../img/Regularization.png)

##### 优化 Optimization

优化的目标是找到损失函数的最低点，即最优解。

核心方法是梯度下降。沿着梯度的反方向向下走，感受当前位置的损失，然后向下迈一步。Follow the slope。

**数值梯度与解析梯度**

数值梯度利用极限定义，取极小的 $h$ 来近似计算梯度。计算慢，但容易实现，常用于梯度检查。

$$
\frac{df}{dx} \approx \frac{f(x+h) - f(x-h)}{2h}
$$

解析梯度通过微积分推导得出，计算精确且快速，是反向传播使用的真正方法。

梯度检查用于验证解析梯度是否实现正确。

损失函数通常是可微的。对于凸函数，局部最小值就是全局最小值。

**梯度下降**

定好迭代次数，或者等待损失收敛。

##### 随机梯度下降 SGD

每次迭代只使用一小批数据来估计梯度。

SGD的问题在于：

1. 鞍点。梯度为0，容易卡住。
2. 噪声。子采样带来的梯度估计噪声，导致更新方向震荡。

##### SGD + Momentum

引入动量，对噪声进行平均，抑制震荡。

更新公式为：

$$
v = \rho v - \alpha \nabla L
$$

$$
x = x + v
$$

其中 $\rho$ 是动量系数，通常取 0.9 左右。$\alpha$ 是学习率。

动量能让收敛可能更慢，但容易找到更优的极小值。因为积累了历史速度，可能会在最小值附近超调，但通过后续调整最终能稳定下来。

In [ ]:
# gradient descent

def compute_gradient(x ,batch_data):
    # Compute the gradient of the loss function with respect to x
    # This is a placeholder function; replace with actual gradient computation
    return np.random.randn(*x.shape)  # Example: random gradient for demonstration


# SGD + Momentum

vx = 0  # Initialize velocity as zero vector
while True:
    dx = compute_gradient(x, batch_data)
    # Placeholder: returns a random array with the same shape as x
    # In practice, this should be the real gradient (e.g., 2*x for f(x)=x^2)

    vx = rho * vx + dx  # v = rho * v + gradient
    x -= learning_rate * vx



##### RMSProp

RMSProp 的核心思路是：**少往陡峭的地方走，多往平坦的地方走。**

它通过计算梯度平方的指数衰减平均，来自适应地调整每个参数的学习率。

更新公式为：

$$
s = \beta s + (1 - \beta) (\nabla L)^2
$$

$$
x = x - \alpha \frac{\nabla L}{\sqrt{s} + \epsilon}
$$

其中 $\beta$ 是衰减率，通常取 0.9 或 0.99。

当梯度大（陡峭）时，$s$ 大，分母大，实际步长变小。当梯度小（平坦）时，$s$ 小，分母小，实际步长变大。

##### Adam (almost)

Adam 本质上是 RMSProp 和 Momentum 的结合。

它同时计算梯度的一阶矩估计（动量）和二阶矩估计（梯度平方），并引入**偏差校正**。

偏差校正解决了初始步长过大的问题。因为在训练初期，$m$ 和 $v$ 初始化接近 0，如果不校正，更新量会非常小。校正后，早期更新步长被放大，模型能快速启动。

Adam 中动量计算时的梯度只看数据损失，最后加上正则化。完整的梯度为：

$$
g_t = \nabla f(w_t) + \lambda w_t
$$

其中 $\lambda w_t$ 是正则化项。

更新公式为：

$$
m = \beta_1 m + (1 - \beta_1) g_t
$$

$$
v = \beta_2 v + (1 - \beta_2) g_t^2
$$

$$
\hat{m} = \frac{m}{1 - \beta_1^t}, \quad \hat{v} = \frac{v}{1 - \beta_2^t}
$$

$$
x = x - \alpha \frac{\hat{m}}{\sqrt{\hat{v}} + \epsilon}
$$

##### AdamW

标准的 Adam 在处理正则化时存在缺陷。因为 L2 正则化被加进了梯度里，它会和自适应学习率交互，导致正则化效果被削弱。

AdamW 的核心是**解耦权重衰减（Decoupled Weight Decay）**。它将权重衰减从梯度更新中分离出来，直接作用于权重本身。

更新公式为：

$$
x = x - \alpha \left( \frac{\hat{m}}{\sqrt{\hat{v}} + \epsilon} + \lambda x \right)
$$

**关于“跑固定轮次后，学习率除以10”**：这不是 AdamW 的专利，而是**步长衰减（StepLR）**。这是一种通用的学习率调度策略，通常每跑固定轮次后，学习率乘以 0.1。它和 AdamW 是正交的概念，可以搭配使用。

##### 余弦学习率衰减

学习率遵循余弦曲线，从初始值平滑衰减到 0 或一个小值。

$$
\alpha_t = \frac{1}{2} \alpha_0 \left( 1 + \cos\left( \frac{t \pi}{T} \right) \right)
$$

其中 $T$ 是总轮次。相比 StepLR，余弦衰减更平滑，后期学习率极小，有助于模型精细收敛。

##### 线性预热

在训练刚开始时，模型权重是随机的，梯度可能很大。如果直接用大学习率，容易导致训练不稳定。

线性预热在最初的几个轮次里，将学习率从 0 或一个极小值线性增加到初始学习率。

```python
# Linear Warmup (pseudocode)
if current_step < warmup_steps:
    lr = base_lr * current_step / warmup_steps
else:
    lr = base_lr
```

##### 线性缩放定律

这是一个经验法则。当批量大小乘以 $k$ 时，学习率也应该乘以 $k$（或 $\sqrt{k}$）。这有助于在大批量训练时保持梯度更新的方差大致恒定。

##### 海森矩阵与大模型

海森矩阵是二阶导数矩阵，能提供损失函数的曲率信息，理论上能帮助优化器找到更好的下降方向。

但在大模型中基本不用。因为参数量巨大，海森矩阵的维度是平方级别，计算和存储成本极高。即使是对角近似，也极其昂贵。因此，大模型训练几乎全部依赖 Adam、AdamW 等一阶优化方法。

In [ ]:
import numpy as np

# Adam Optimizer

first_moment = 0        # m: first moment estimate 
#first_moment = np.zeros_like(x)

second_moment = 0       # v: second moment estimate

beta1 = 0.9             # Decay rate for first moment
beta2 = 0.999           # Decay rate for second moment
learning_rate = 0.001
epsilon = 1e-8

for t in range(1, num_iterations + 1):
    dx = compute_gradient(x)    # g_t: gradient at current step
    
    # Update biased first moment estimate (m = beta1 * m + (1 - beta1) * g)
    first_moment = beta1 * first_moment + (1 - beta1) * dx
    
    # Update biased second raw moment estimate (v = beta2 * v + (1 - beta2) * g^2)
    second_moment = beta2 * second_moment + (1 - beta2) * (dx ** 2)
    
    # Compute bias-corrected first moment estimate
    first_moment_corrected = first_moment / (1 - beta1 ** t)
    
    # Compute bias-corrected second raw moment estimate
    second_moment_corrected = second_moment / (1 - beta2 ** t)
    
    # Update parameters (x = x - lr * m_hat / (sqrt(v_hat) + epsilon))
    x -= learning_rate * first_moment_corrected / (np.sqrt(second_moment_corrected) + epsilon)


In [ ]:
# AdamW (Decoupled Weight Decay) 解耦权重衰减,修改最后一行
x -= learning_rate * (first_moment_corrected / (np.sqrt(second_moment_corrected) + epsilon) + weight_decay * x)

##### SVM损失函数

**用途**

SVM损失是数据损失的核心组成部分，用于衡量模型输出的预测分数与真实标签之间的差距。它通常用于线性分类器或神经网络的输出层，为模型提供优化方向。

**核心原理：安全边界与合页损失**

SVM的核心思想是：不仅要预测正确，还要自信地预测正确。它希望正确类别的分数，比其他所有错误类别的分数，至少高出一个安全边界，即 Margin，通常设为 $\Delta = 1.0$。

如果正确类别的分数比某个错误类别的分数高出至少 $\Delta$，模型就认为在这个类别上已经足够安全，不再产生损失，即 Loss = 0。否则，就会产生惩罚，即 Loss > 0。这种机制被称为合页损失 Hinge Loss。

**数学公式**

对于第 $i$ 个样本，多类SVM损失的公式为：

$$
L_i = \sum_{j \neq y_i} \max(0, s_j - s_{y_i} + \Delta)
$$

其中 $s_j$ 是模型对第 $j$ 个错误类别的预测分数，$s_{y_i}$ 是模型对真实类别 $y_i$ 的预测分数，$\Delta$ 是安全边界，通常取 1.0。$\max(0, \cdot)$ 就是合页损失，如果括号内小于0，即已经足够安全，则损失为0。

整个数据集的平均SVM损失为：

$$
L = \frac{1}{N} \sum_{i=1}^N L_i + \lambda R(W)
$$

其中 $\lambda R(W)$ 是正则化项。

**直观例子**

假设有3个类别，分别是猫、狗、汽车，真实标签是猫，即 $y_i = 0$。模型的预测分数为：猫 3.2，狗 5.1，汽车 -1.7。设边界 $\Delta = 1.0$。

对于错误类别狗，$\max(0, 5.1 - 3.2 + 1.0) = \max(0, 2.9) = 2.9$，产生损失。

对于错误类别汽车，$\max(0, -1.7 - 3.2 + 1.0) = \max(0, -3.9) = 0$，不产生损失。

所以这个样本的损失为 2.9 + 0 = 2.9。

这说明模型虽然预测对了猫，但只比狗高了1.9分，没有达到安全边界1.0的自信差距，因此被罚款。

**SVM损失的作用**

指导优化方向。它为模型提供了一个可微的梯度信号，驱动模型去拉大正确类别与错误类别之间的分数差距。

控制模型行为。边界 $\Delta$ 决定了模型要多自信才算满意。$\Delta$ 越大，模型被迫拉开的分数差距就越大。

与Softmax的对比。SVM只关心分数是否超过边界，不关心分数之间的绝对差异。而Softmax，也就是交叉熵，则会把分数转化为概率分布，关心正确类别的概率有多大。这是两种不同的优化哲学。

---

#### 神经网络与反向传播

##### SVM损失函数
SVM损失函数通常用于线性分类器，用来衡量预测分数与真实分数之间的差异。它属于数据损失的一部分。

##### 线性映射
线性分类器的基础公式是：

$$
f = Wx
$$

这种模型只能解决线性可分的问题。

##### 两层神经网络
在线性模型的基础上引入隐藏层，得到两层神经网络：

$$
f = W_2 \max(0, W_1 x)
$$

其中 $W_1$ 是第一层权重，$W_2$ 是第二层权重。$\max(0, \cdot)$ 即 ReLU 激活函数。完整形式通常会加上偏置项 $b_1$ 和 $b_2$：

$$
f = W_2 \max(0, W_1 x + b_1) + b_2
$$

##### 激活函数
激活函数引入非线性机制。如果没有激活函数，多层神经网络无论叠多深，本质上仍然等价于一个线性变换。

**ReLU**

ReLU 即整流线性单元。公式为：

$$
f(x) = \max(0, x)
$$

计算简单，收敛速度快，是当前最常用的默认激活函数。

**死神经元**

当某个神经元的权重使得其对所有输入都输出负数时，ReLU 的梯度为 0。该神经元将永久失活，不再更新，这叫死神经元问题。

**Leaky ReLU**

为了解决死神经元问题，Leaky ReLU 在负半轴引入一个小斜率 $\alpha$：

$$
f(x) = \max(\alpha x, x)
$$

通常 $\alpha$ 取 0.01 左右。

**ELU**

ELU 即指数线性单元，在负半轴使用指数函数：

$$
f(x) = \begin{cases} x & x > 0 \\ \alpha(e^x - 1) & x \le 0 \end{cases}
$$

负半轴均值接近 0，有助于加速收敛，但计算量稍大。

**etc...**

还有其他变体，例如 GELU、Swish、Maxout 等。

##### 制造非线性
激活函数的作用就是制造非线性。没有非线性，再深的网络也只是线性模型。

##### 全连接神经网络
全连接神经网络也叫多层感知机 MLP。每一层的每个神经元都与前一层的所有神经元相连。通过堆叠多个全连接层并配合激活函数，网络可以拟合极其复杂的非线性函数。

例如，在一个简单的线性分类器中，前向传播可能仅仅是：

$$
f = W x + b
$$

而在一个简单的两层神经网络中，前向传播引入了激活函数，可能如下：

$$
f = W_2 \max(0, W_1 x + b_1) + b_2
$$



In [ ]:
# forward pass 
import numpy as np

def forward_pass(x, W1, b1, W2, b2):
    """
    Simple forward pass for a 2-layer neural network.
    x: input data, shape (N, D)
    W1, b1: weights and bias of the first layer
    W2, b2: weights and bias of the second layer
    Returns the output scores.
    """
    # First layer: linear transformation followed by ReLU activation
    hidden = np.maximum(0, x @ W1 + b1)
    
    # Second layer: linear transformation to output scores
    scores = hidden @ W2 + b2
    
    return scores

In [ ]:
# a 2-layer neural network 

import numpy as np

# Define network dimensions
N, D_in, H, D_out = 64, 1000, 100, 10

# Randomly initialize input data, target, and weights
x = np.random.randn(N, D_in)
y = np.random.randn(N, D_out)
w1 = np.random.randn(D_in, H)
w2 = np.random.randn(H, D_out)

learning_rate = 1e-4

for t in range(500):
    # Forward pass: compute predicted y
    # First layer: linear transformation followed by Sigmoid activation
    h = 1 / (1 + np.exp(-x.dot(w1)))
    # Second layer: linear transformation to output
    y_pred = h.dot(w2)

    # Compute loss using squared error
    loss = np.square(y_pred - y).sum()
    if t % 100 == 0:
        print(f"Iteration {t}, Loss: {loss}")

    # Backward pass: compute gradients
    # Gradient of loss with respect to y_pred
    grad_y_pred = 2.0 * (y_pred - y)
    # Gradient of loss with respect to w2
    grad_w2 = h.T.dot(grad_y_pred)
    # Gradient of loss with respect to hidden layer output h
    grad_h = grad_y_pred.dot(w2.T)
    # Add Sigmoid derivative: multiply by h * (1 - h)
    grad_h = grad_h * h * (1 - h)
    # Gradient of loss with respect to w1
    grad_w1 = x.T.dot(grad_h)

    # Update weights using gradient descent
    w1 -= learning_rate * grad_w1
    w2 -= learning_rate * grad_w2


##### 正则化与网络规模的区别

**正则化**用于限制模型权重的大小，防止过拟合，比如 **L1 和 L2 正则化**。**网络规模**指层数和神经元数量，属于模型架构设计。不能用**缩小网络规模**来替代正则化。缩小网络会直接降低模型的表示能力，容易导致**欠拟合**，而正则化是在保持模型容量的前提下约束参数。

##### 神经元与激活函数

神经网络中的每个神经元可以视为计算图中的一个**门单元**。**激活函数**是一类特殊的门，引入**非线性**。前向传播时，它将输入映射为输出。反向传播时，它根据输入和输出计算**局部梯度**，并将**上游梯度**传递给**下游梯度**。

##### 计算图

![计算图](../img/backpropagation.png)

**计算图**将复杂函数拆解为一系列简单的中间步骤。**前向传播**时，依次计算中间节点的输出并保存。**反向传播**时，利用**链式法则**从输出层向输入层逐级求导。

##### 反向传播的目标与流程

反向传播的目的是计算**损失 $L$** 对所有变量包括**权重 $W$** 和**偏置 $b$** 的梯度。这些梯度将被**优化器**用来更新权重。

在实际框架中，不会为每一层手动写出导数函数，而是通过逐级反向传播自动完成梯度计算。

反向传播的完整流程如下。

1. **前向传播**求出每一步的中间输入和输出，保存在内存中供后续求导使用。
2. **反向传播**从末端开始。**末端梯度**通常恒为 1，即 $\frac{\partial L}{\partial L} = 1$。
3. 每一步先求**局部梯度**，再乘以**上游梯度**，得到**下游梯度**，并将其继续向下游传递。

##### 门单元及其梯度传播规则

**加法门**是**梯度分配器**。对于 $z = x + y$，局部梯度 $\frac{\partial z}{\partial x} = 1$，$\frac{\partial z}{\partial y} = 1$。加法门将上游梯度原封不动地分发给两个输入。

**乘法门**是**梯度交换器**。对于 $z = x \cdot y$，局部梯度 $\frac{\partial z}{\partial x} = y$，$\frac{\partial z}{\partial y} = x$。乘法门把上游梯度乘以另一个输入的值后再分发。

**复制门**用于把一个变量复制到多个分支。反向传播时，来自不同分支的梯度会在复制点进行累加，因为同一个变量对多个输出都有贡献。

**最大值门**是**梯度路由器**。对于 $z = \max(x, y)$，梯度只会传递给前向传播中数值较大的那个输入，另一个输入的梯度为 0。

##### Sigmoid 门

**Sigmoid 函数**作为激活函数时，其局部梯度为 $\sigma'(z) = \sigma(z)(1 - \sigma(z))$。反向传播时，上游梯度乘以这个局部梯度得到下游梯度。由于 Sigmoid 的导数最大值仅为 0.25，在深层网络中使用会导致梯度逐层衰减，产生**梯度消失问题**。


In [ ]:
import numpy as np

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

# 前向传播
def forward_pass(w0, x0, w1, x1, w2):
    s0 = w0 * x0        # 乘法门
    s1 = w1 * x1        # 乘法门
    s2 = s0 + s1        # 加法门
    s3 = s2 + w2        # 加法门
    L = sigmoid(s3)     # Sigmoid 门
    return s0, s1, s2, s3, L

# 前向传播计算
w0, x0, w1, x1, w2 = 2.0, 3.0, 1.0, 4.0, 0.5
s0, s1, s2, s3, L = forward_pass(w0, x0, w1, x1, w2)

# 反向传播
grad_L = 1.0            # 末端梯度恒为 1，即 dL/dL
grad_s3 = grad_L * (L * (1 - L))    # Sigmoid 门：乘以局部梯度 L*(1-L)

grad_s2 = grad_s3 * 1.0             # 加法门：梯度原样分发
grad_w2 = grad_s3 * 1.0             # 加法门：梯度原样分发

grad_s0 = grad_s2 * 1.0             # 加法门：梯度原样分发
grad_s1 = grad_s2 * 1.0             # 加法门：梯度原样分发

grad_w0 = grad_s0 * x0              # 乘法门：乘以另一个输入 x0
grad_x0 = grad_s0 * w0              # 乘法门：乘以另一个输入 w0

grad_w1 = grad_s1 * x1              # 乘法门：乘以另一个输入 x1
grad_x1 = grad_s1 * w1              # 乘法门：乘以另一个输入 w1

print(f"L: {L}")
print(f"grad_w0: {grad_w0}, grad_x0: {grad_x0}")
print(f"grad_w1: {grad_w1}, grad_x1: {grad_x1}")
print(f"grad_w2: {grad_w2}")

In [ ]:
import torch

# 前向传播和反向传播的接口
class Multiply(torch.autograd.Function):
    @staticmethod
    def forward(ctx, input1, input2):
        # ctx 是上下文对象，用于在前向和反向传播之间共享数据
        # 保存输入张量，反向传播时需要用它们计算局部梯度
        ctx.save_for_backward(input1, input2)
        # 前向计算：返回两个输入的乘积
        return input1 * input2

    @staticmethod
    def backward(ctx, grad_output):
        # grad_output 是损失函数对 forward 输出结果的梯度（上游梯度）
        # 从上下文中取出前向传播时保存的输入张量
        input1, input2 = ctx.saved_tensors
        # 乘法门的局部梯度：对 input1 的梯度等于 grad_output 乘以 input2
        grad_input1 = grad_output * input2
        # 对 input2 的梯度等于 grad_output 乘以 input1
        grad_input2 = grad_output * input1
        # 返回损失对两个输入的梯度，顺序必须与 forward 的输入顺序一致
        return grad_input1, grad_input2



vector to vector

**向量对向量求导**

当函数输入是向量，输出也是向量时，对输出向量 $\mathbf{y}$ 的每个元素关于输入向量 $\mathbf{x}$ 的每个元素求偏导，得到雅可比矩阵。

$$
\mathbf{J} = \frac{\partial \mathbf{y}}{\partial \mathbf{x}} = 
\begin{bmatrix}
\frac{\partial y_1}{\partial x_1} & \cdots & \frac{\partial y_1}{\partial x_n} \\
\vdots & \ddots & \vdots \\
\frac{\partial y_m}{\partial x_1} & \cdots & \frac{\partial y_m}{\partial x_n}
\end{bmatrix}
$$

**损失函数 $L$ 是标量**

当最终损失 $L$ 是一个标量时，$L$ 对向量或矩阵求导的结果称为梯度，其形状与被求导的变量完全相同。比如 $L$ 对矩阵 $W$ 的梯度 $\frac{\partial L}{\partial W}$，其形状与 $W$ 一致。

**矩阵乘法的反向传播**

对于矩阵乘法 $y = xW$，输入 $x$ 是 $N \times D$ 的矩阵，权重 $W$ 是 $D \times M$ 的矩阵，输出 $y$ 是 $N \times M$ 的矩阵。

在反向传播时，不需要构造巨大的雅可比矩阵。$x$ 的第 $i$ 行只影响 $y$ 的第 $i$ 行。上游梯度 $\frac{\partial L}{\partial y}$ 的形状是 $N \times M$。根据链式法则，可以通过矩阵乘法直接求出对 $x$ 和 $W$ 的梯度：

$$
\frac{\partial L}{\partial x} = \frac{\partial L}{\partial y} W^T
$$

$$
\frac{\partial L}{\partial W} = x^T \frac{\partial L}{\partial y}
$$

为矩阵乘法写反向传播函数，利用矩阵乘法计算梯度，避免了逐个元素计算雅可比矩阵。

```python
import numpy as np

class MatMul:
    def __init__(self):
        self.x = None
        self.W = None

    def forward(self, x, W):
        # 保存前向传播的输入，反向传播计算局部梯度时需要用到
        self.x = x
        self.W = W
        # 前向计算：矩阵乘法
        return x.dot(W)

    def backward(self, dout):
        # dout 是上游传下来的梯度，形状为 (N, M)
        # 损失对输入 x 的梯度，形状为 (N, D)
        dx = dout.dot(self.W.T)
        # 损失对权重 W 的梯度，形状为 (D, M)
        dW = self.x.T.dot(dout)
        return dx, dW
```


**Recap**

用线性分类器解决图像分类，用张量定义输入输出，用权重矩阵 $W$ 预测损失得分，用损失函数判断 $W$ 的表现。线性分类并不强大，因此提出了神经网络，堆叠线性与非线性层。为了优化分类器，需要计算更复杂的 $W$，因此引入了计算图、梯度和反向传播。

机制上，可以定义各种节点，都遵循计算输出和局部梯度的接口。只要所有节点都遵循这个规则，就能组合成能进行任意计算的复杂大图。上游梯度的形状始终和输出完全一样，下游梯度是损失对输入的导数，形状和输入一样。反向传播的核心在于链式法则，通过将局部梯度逐级向上传递，自动求出损失对每一个参数的梯度，这为后续的梯度下降更新提供了依据。

##### 特征表示

传统方法是人工定义一种表示作为输入，比如颜色直方图、方向梯度直方图 HOG，但这类方法已被淘汰。这类人工特征提取需要大量专家知识，且泛化能力有限，因为它们丢失了图像的空间信息，无法捕捉复杂的纹理和语义。

现在的趋势是端到端设计，让数学和计算比人类更擅长寻找中间函数。网络通常由卷积层、池化层、非线性层和全连接层 MLP 组成。特征提取器不再是手工设计，而是完全由数据驱动，通过反向传播自动学习得到。这种方式使得网络可以针对具体任务自动提取最适合的特征。

##### 历史

LeNet-5 是最早的卷积神经网络之一，由 Yann LeCun 提出，用于手写数字识别。AlexNet 在 2012 年 ImageNet 竞赛中取得突破，引爆了深度学习，它引入了 ReLU 激活函数和 Dropout 来防止过拟合。随后 VGG 和 ResNet 通过更深的网络结构不断刷新性能，ResNet 更是引入了残差连接解决了深层网络梯度消失的问题。如今 Transformer 架构在部分任务中替代了卷积神经网络，比如 Vision Transformer 将图像分块后送入注意力机制处理。


---

#### 卷积网络

##### 卷积、滤波器与特征图的关系

这三者是卷积层最核心的组成，理解它们的物理关系至关重要。

**滤波器**是一个小的权重矩阵，例如 $3 \times 3$ 大小，它是提取特征的**工具**。滤波器内部的值就是网络需要学习的参数。

**卷积**是滤波器在图像上滑动的**动作**。滤波器按照步幅 $S$ 在图像上滑动，每到一个位置，就将滤波器内的权重与图像对应位置的$K \times K $个像素值进行逐元素相乘并求和，得到一个标量输出。这就是滤波器在该位置的匹配得分。这个计算本质上就是点积，也就是模板匹配。

**特征图**是卷积动作产生的**结果**。当滤波器扫完整张图，所有位置的得分排列成一个二维矩阵，这就是激活图或特征图。特征图上的每一个点，代表输入图像对应位置与滤波器模式的匹配程度。

为了提取多种不同的特征，通常会使用多个滤波器。每个滤波器生成一张特征图，多个滤波器并行工作，所有特征图汇成一个三维张量。单个滤波器的深度必须与输入的通道数 $C_{in}$ 保持一致。如果输入是 RGB 三通道，滤波器就变成 $3 \times 3 \times 3$ 的三维小块，逐通道计算点积后再把三个结果相加。

##### 卷积层的参数与计算

在卷积层中，区分参数和超参数很关键。卷积核数量、尺寸、步幅 $S$ 和填充 $P$ 都是超参数，是训练前定好的。滤波器内部的数值和偏置项是可学习参数，通过反向传播更新。反向传播时，同一个滤波器在不同空间位置的梯度会累加，因为参数共享。参数共享是卷积层高效的核心原因。

输出尺寸由以下公式决定：

$$
H_{out} = \left\lfloor \frac{H_{in} - K + 2P}{S} \right\rfloor + 1
$$

不加填充时，特征图会越来越小，边缘像素容易被忽略。引入 padding 补 0 可以保留边缘信息，保证网络可以不断加深。常见做法是 $K$ 取奇数，$P = (K - 1) / 2$。

卷积层通常批量处理四维张量。

- 输入维度为 $N, C_{in}, H, W$，

- 滤波器组维度为 $C_{out}, C_{in}, K_w, K_h$，

- 输出维度为 $N, C_{out}, H', W'$。

其中 $C_{out}$ 等于滤波器的数量。

在卷积层之间，数据的尺寸和通道变化是有严格规律的。当前层的输出通道数 $C_{out}$ 将直接作为下一层的输入通道数 $C_{in}$。例如，第一层输入通道为 3，设置 32 个滤波器，输出通道即为 32，那么第二层的输入通道必须相应为 32。通道数通常随网络加深而增加，空间尺寸则通过步幅大于 1 的卷积或池化层来主动减小，以扩大感受野并降低计算量。

卷积网络就是包含多个卷积层的计算图。卷积是点积，点积是线性算子，所以卷积层之间必须加激活函数引入非线性，比如ReLU。如果缺少非线性，无论叠多少层卷积，最终依然等价于一个单层线性变换。

在特征层次上，浅层滤波器倾向于学习颜色、边缘等低级特征，深层滤波器则组合这些低级特征，学习纹理和物体部件等高级语义特征。


##### 可学习参数数量

![CNN示例](../img/CNN_exa.png)

以图示为例，输入体积为 $3 \times 32 \times 32$，使用 10 个 $5 \times 5$ 滤波器，步幅 1，填充 2。输出体积为 $10 \times 32 \times 32$。

每个滤波器的参数数量为 $3 \times 5 \times 5 + 1 = 76$，其中 1 是偏置项。10 个滤波器的总参数数量为 $10 \times 76 = 760$。如果使用全连接层处理同样大小的输入输出，参数量将达到数千万级别。参数共享是卷积层高效的核心原因，一个滤波器在图像的不同位置共享相同的权重，极大地减少了参数量。

##### 感受野

![感受野](../img/Receptive_Fields.png)

有效感受野指原始图像中有多少像素能影响到网络后端的某个激活值。感受野越大，网络能看到的全局上下文信息就越多。感受野随层数线性扩大。假设第 $l$ 层的感受野为 $R_l$，卷积核大小为 $K_l$，步幅为 $S_l$，则下一层的感受野递推公式为：

$$
R_{l+1} = R_l + (K_{l+1} - 1) \times \prod_{i=1}^{l} S_i
$$

除了增加层数，还可以通过增大卷积核尺寸或使用步幅卷积来更快地扩大感受野。


##### 池化层

池化层是卷积网络中用于下采样的核心组件。卷积层负责提取特征，池化层负责对特征图进行压缩，通道数保持不变，但空间尺寸会显著减小。这能降低后续计算量，扩大有效感受野，并引入一定的平移不变性。池化层独立地在每个通道上操作。

核心思路是对图像的高宽维度做合并处理。

根据计算方式的不同，池化层主要分为以下几种类型。

**最大池化 Max Pooling**

这是最常用的池化方式。在每个池化窗口内，取出数值最大的那个元素，作为输出特征图对应位置的值。

举例来说，假设输入是一个 $4 \times 4$ 的单通道特征图，池化窗口大小为 $2 \times 2$，步长为 $2$。左上角 $2 \times 2$ 的窗口内如果有数值 $1, 3, 2, 5$，最大池化会输出 $5$。依次滑动窗口，$4 \times 4$ 的输入就被压缩成 $2 \times 2$ 的输出，空间尺寸减半。

最大池化保留了窗口内最强烈的激活值，相当于提取了局部最显著的特征，保留了纹理细节。在反向传播时，它只把上游梯度传给前向传播中数值最大的那个位置，其余位置的梯度为 0，这保证了梯度只通过最强激活的路径流动。

**平均池化 Average Pooling**

在每个池化窗口内，取所有元素的平均值，作为输出特征图对应位置的值。

以同样的 $2 \times 2$ 窗口为例，如果窗口内数值为 $1, 3, 2, 5$，平均池化会输出 $(1+3+2+5)/4 = 2.75$。

平均池化相当于对局部区域做了平滑，保留了整体背景信息。在反向传播时，它把上游梯度平均分配到窗口内的每一个位置，相当于每个输入元素都承担了均等的梯度贡献。平均池化是线性算子，而最大池化是非线性的。

**全局平均池化 Global Average Pooling**

这是一种特殊的平均池化。池化窗口的大小等于整个特征图的空间尺寸，直接把每个通道上的所有值求平均，输出一个数值。

全局平均池化常用于替代网络末端的全连接层。例如，输入特征图的维度是 $N, C, H, W$，经过全局平均池化后，输出维度变成 $N, C, 1, 1$。这能极大地减少参数量，防止过拟合，在现代网络如 ResNet 中被广泛使用。

**其他池化变体**

除了上述三种，还有一些变体用于特定场景。比如**随机池化 Stochastic Pooling**，按照概率大小随机选择窗口内的元素，这也是一种正则化手段。还有**混合池化 Mixed Pooling**，结合最大池化和平均池化，取其加权平均。

**池化层的特性与机制**

- 池化层没有可学习的参数，因为怎么池化只是一个超参数，它只是进行固定的数学运算。池化层通常不需要填充，也不需要 ReLU 激活函数，这属于抗混叠下采样，可以防止下采样过程中的信息混叠。

- 需要注意的是，虽然池化层没有参数，但依然需要反向传播，以便将梯度传回给前面的卷积层。如果使用最大池化，梯度只传给最强激活的位置。如果使用平均池化，梯度均匀分配到窗口内的所有位置。

##### 平移等变性与平移不变性

平移等变性指的是平移和卷积的顺序不重要。数学上表示为：

$$
f(\text{translate}(x)) = \text{translate}(f(x))
$$

这意味着输入图像平移后，输出的特征图也会平移相同的量，但特征值保持不变。这是卷积操作自身的性质，参数共享使得滤波器在平移后依然能捕捉到相同的特征。

而池化层则引入了平移不变性。物体在图像中稍微移动，经过池化下采样后，最大激活值可能依然被保留，最终的分类结果保持不变。两者概念不同，需要区分：卷积是等变的，池化提供了一定的不变性，这共同增强了网络对目标位置变化的鲁棒性。

---





#### 卷积神经网络的训练以及 CNN 架构

##### 怎么构建 CNN

构建一个卷积神经网络，通常是由卷积层、池化层、归一化层、激活函数以及全连接层堆叠而成。卷积层负责提取特征，池化层负责下采样，归一化层负责稳定训练，激活函数引入非线性，全连接层负责最终的分类或回归。

**归一化层**

归一化层的作用是将数据转换为单位高斯分布，通过缩放和平移操作，让每一层的输入分布保持稳定，从而加速训练。主要的区别在于**怎么计算均值和标准差**。

在卷积神经网络中，输入通常是四维张量 $N, C, H, W$。不同的归一化方法在不同的维度上计算统计量：

- **批量归一化 Batch Normalization**：在批次维度 $N$ 和空间维度 $H, W$ 上计算均值和方差，对每个通道 $C$ 独立进行。它要求批次大小不能太小，否则统计量不准确。
- **层归一化 Layer Normalization**：在通道维度 $C$ 和空间维度 $H, W$ 上计算，对每个样本 $N$ 独立进行。它不依赖于批次大小，常用于循环神经网络和 Transformer。
- **实例归一化 Instance Normalization**：只在空间维度 $H, W$ 上计算，对每个样本的每个通道独立进行。常用于风格迁移任务。

![归一化层](../img/Normalization_Layers.png)

**Dropout**

Dropout 是一种在训练时引入随机性的正则化手段。它通过设定一个固定的超参数，即丢弃概率，在每次前向传播时随机将一部分神经元的输出置为 0。

这可以使用掩码技巧来实现，被丢弃的部分不参与本轮的前向和反向计算。

直觉上，这能迫使网络不依赖特定的神经元，从而学习到更泛化的特征。



```python
# Vanilla Dropout: Not recommended implementation (see notes below)
p = 0.5 # probability of keeping a unit active. higher = less dropout

def train_step(X):
    # forward pass for example 3-layer neural network
    H1 = np.maximum(0, np.dot(W1, X) + b1)
    U1 = np.random.rand(*H1.shape) < p # first dropout mask
    H1 *= U1 # drop!
    H2 = np.maximum(0, np.dot(W2, H1) + b2)
    U2 = np.random.rand(*H2.shape) < p # second dropout mask
    H2 *= U2 # drop!
    out = np.dot(W3, H2) + b3

def predict(X):
    # ensembled forward pass
    H1 = np.maximum(0, np.dot(W1, X) + b1) * p # NOTE: scale the activations
    H2 = np.maximum(0, np.dot(W2, H1) + b2) * p # NOTE: scale the activations
    out = np.dot(W3, H2) + b3
```



需要注意的是，测试时不再丢弃任何值。因为训练时按概率 $p$ 随机丢弃，测试时全部保留，这会导致测试时的输入量级比训练时大。为了保证输入量级一致，需要在测试时给激活值乘上 $p$，也就是上面代码中 `* p` 的操作。

（注：现代深度学习框架如 PyTorch 采用的是反向 Dropout。在训练时就将保留的神经元除以 $p$，测试时不做任何处理。这样可以在测试时保持原有前向传播逻辑不变，且只需在训练时缩放一次。）

**激活函数的选择**

早期常用 Sigmoid 作为激活函数。但 Sigmoid 在极负极正区域梯度趋近于 0，随着层数增多，反向传播的梯度会越来越小，产生梯度消失问题，因此 Sigmoid 不再被经常使用。

现在更多使用 ReLU。ReLU 收敛快，但仍有问题：任何负输入都会导致输出为 0，可能产生死神经元。

为了改进，引入了 GELU。GELU 是 Transformer 里主要使用的激活函数，在极端情况下也会逼近 ReLU，但其在零点附近是平滑的，使得优化过程更加稳定。

无论选择哪种，激活函数总是用在线性层（卷积层或全连接层）之后。

##### CNN 的组合架构

**VGGNet**

VGGNet 探索了网络深度对性能的影响。它有一个重要的发现：为什么 3x3 卷积层（步长为 1）有效？

因为堆叠三个 3x3 卷积层的感受野等同于一个 7x7 卷积层，但三个 3x3 层的参数量更少（$3 \times (3^2 C^2) = 27C^2$ 对比 $7^2 C^2 = 49C^2$），且由于中间加入了非线性激活函数，使其建模能力更加复杂，能提取更抽象的特征。

**ResNets**

ResNets 试图解决一个核心问题：叠加更深的层会怎么样？

理论上，深层网络能囊括浅层网络的所有模型，但实验发现，深层网络更难优化，因此可能表现更差。这被称为退化问题，深层网络难以逼近浅层模型的水平，单靠时间无法达到。

ResNets 引入了残差连接。它让网络拟合残差映射，即 $F(x) + x$。当恒等映射是最优解时，网络只需要将残差 $F(x)$ 的权重学习为 0，就能轻松实现恒等映射。直觉上，加入残差连接相当于给梯度提供了一条高速公路，使得深层网络能够更容易地学习恒等映射，从而在极深的网络中也能保持优秀的性能。

##### 权重初始化

**Kaiming 初始化**

权重初始化对训练至关重要。如果权重太小，激活值会迅速衰减到 0；如果权重太大，激活值会迅速爆炸。为了保持每一层激活值的方差稳定，Kaiming 初始化引入了 ReLU 修正。

```python
dims = [4096] * 7
hs = []
x = np.random.randn(16, dims[0])
for Din, Dout in zip(dims[:-1], dims[1:]):
    W = np.random.randn(Din, Dout) * np.sqrt(2/Din) # ReLU correction
    x = np.maximum(0, x.dot(W))
    hs.append(x)
```

通过乘以 $\sqrt{2 / D_{in}}$，激活值的分布能被很好地缩放。在实验中，使用 Kaiming 初始化后，各层激活值的均值和标准差能保持稳定，实现了均值和标准差不随层数加深而剧烈变化的理想效果。

![Kaiming 初始化](../img/Kaiming_Init.png)

对比实验：
- 权重值过小：`W = 0.01 * np.random.randn(...)`，深层网络的激活值趋近于 0。
- 权重值过大：`W = 0.05 * np.random.randn(...)`，激活值迅速爆炸。

![WI1](../img/Weight_Init1.png)
![WI2](../img/Weight_Init2.png)
- 归一化层也能在某种程度上解决激活值爆炸的问题。

##### 怎么训练 CNN

**数据处理与数据增强**

训练 CNN 的第一步是数据处理。通常需要对图像进行归一化，计算整个数据集的均值和标准差，比如使用 ImageNet 的统计量。

数据增强是防止过拟合的重要手段。可以做哪些增强？
- 翻转、缩放、裁剪。需要让人类依然能辨认，但模型更难死记硬背。
- 测试时增强：对同一张图做多种增强，平均预测结果。
- 色彩抖动、亮度变化。
- Cutout（随机裁剪区域置 0）。

**迁移学习 Transfer Learning**

在实际应用中，我们往往没那么多数据。此时可以使用迁移学习。

![迁移学习](../img/Transfer_CNN.png)

如果数据量较小，与 ImageNet 数据相似：换掉 ImageNet 的最后一层全连接层（输出类别数改为自己的类别数），冻结其他层。这相当于把预训练模型当作固定的特征提取器，只训练最后的分类器。

如果数据量较大，可以微调所有层：使用预训练模型初始化，然后微调整个网络。数据越大，就越有利于训练更多的层。

**超参数选择**

超参数选择有一套实用的流程：
1. 小样本过拟合测试，看损失能否快速降低。这通常用于检查代码是否跑通，以及模型是否有能力拟合数据。
2. 粗略的超参数网格，看损失和准确率曲线。
3. 直到训练准确率开始偏离验证准确率，这表明开始过拟合。
4. 过程可以反复进行。
5. 在超参数空间里随机搜索，比如随机选择学习率、正则化强度等。

（注：随机搜索通常比网格搜索更高效，因为并非所有超参数都同等重要。）


---



超参数选择没有绝对完美的固定公式，它高度依赖于具体任务、网络架构和数据集。一般通过反复的实验迭代来寻找最优组合。整套流程可以按照以下阶段逐步展开，并配合监控指标进行动态调整。

**代码正确性验证**

在开始正式的调参之前，一般需要先验证代码逻辑是否无误。

- 若在小样本数据上，训练损失无法快速降低，甚至无法让模型过拟合，则说明代码本身很可能存在缺陷。需要检查数据加载、网络连接、损失函数或者学习率是否过大导致梯度爆炸。
- 若一切正常，训练损失能顺利下降，则进入下一阶段。

**观察曲线变化并对应调整**

在确定了代码没问题之后，通常先使用少量 Epoch 进行粗略的超参数网格搜索，通过观察训练损失和验证准确率曲线的变化趋势，进行针对性调整。这是调参最核心的环节。

- 若训练损失和验证损失都居高不下，模型一般处于欠拟合状态。此时需要增大模型容量，比如增加层数或滤波器数量，或者减少正则化强度，并检查学习率是否过小。
- 若训练损失持续下降，而验证损失开始上升，模型一般进入过拟合状态。此时需要增加数据增强，增大正则化强度，如权重衰减或 Dropout 概率，或者使用早停策略。
- 一旦训练准确率开始偏离验证准确率，且曲线出现明显分叉，一般标志着过拟合开始发生。此时需要及时记录当前的最佳模型权重，并回退到分叉前的超参数设置。
- 若损失曲线出现剧烈震荡甚至发散，一般说明学习率过大。此时需要立即减小学习率，比如将其缩小为原来的十分之一再尝试。

**搜索策略**

在确定了合理的超参数搜索范围后，一般推荐在超参数空间里进行随机搜索。

- 网格搜索会均匀地在网格上取点，但实际上某些超参数对结果的影响远大于另一些。随机搜索能在不考虑网格布局的情况下，更密集地探索对结果影响更大的那个维度，从而在相同的计算预算下找到更优的解。
- 一般通过比较不同超参数组合的最终验证集指标来进行筛选。

**超参数调整的优先级**

超参数的调整本身也有优先级。按照重要性从高到低，依次为：

- 学习率：一般是最重要的超参数。通常需要观察验证集的收敛速度与最终精度。一般建议在 log 空间上进行搜索，例如从 0.1 到 0.0001 之间进行对数均匀采样。
- 正则化强度：一般包括权重衰减系数和 Dropout 概率。通常需参考训练集和验证集准确率之间的差距。若差距过大，一般需增大正则化强度；若整体分数偏低，则可适当减小。
- 批量大小：通常与学习率存在线性缩放关系。当批量大小翻倍时，学习率一般也应当适当调大，以保证梯度更新的方差大致恒定。
- 网络架构超参数：比如层数、滤波器数量和卷积核尺寸。一般在前几项调优完成后再进行微调，通常需观察模型容量是否匹配任务复杂度。

**早停策略**

早停是在实践中非常有效的策略。在验证集准确率连续多个 Epoch 不再提升时，一般应立即停止训练，并保存验证集上表现最好的那组权重。这不仅能节省训练时间，还能防止模型在训练集上过度记忆。

---


#### 序列建模与循环神经网络 RNN

##### 序列建模 Sequence Modeling

在处理图像分类任务时，模型通常是对单张图片进行独立预测。但在许多任务中，数据之间存在时间或顺序上的关联，比如自然语言文本、音频信号、视频帧序列。这些数据通常被称为序列数据，而序列建模的任务就是让模型能够处理这类拥有前后依赖关系的数据。

通常，序列建模需要模型具备“记忆”能力，能够根据之前时刻的信息来影响当前时刻的输出。循环神经网络正是为了解决这类问题而提出的经典架构。

##### RNN 的核心机制

![RNN](../img/RNN.png)

RNN 通过引入隐藏状态来保存历史信息。隐藏状态就像是网络的“记忆”，它会随着时间步的推进而更新，将过去的信息传递到未来。

RNN 的展开结构如上图所示。输入序列为 $x_1, x_2, ..., x_t$，网络在每个时间步接收一个输入，并结合上一时间步的隐藏状态，计算当前的隐藏状态和输出。这种结构被称为展开的 RNN。

RNN 的递推公式如下：

$$
h_t = f_W(h_{t-1}, x_t)
$$

$$
y_t = f_y(W_{hy} h_t)
$$

其中 $h_t$ 是当前时间步的隐藏状态，$h_{t-1}$ 是上一时间步的隐藏状态，$x_t$ 是当前时间步的输入。$y_t$ 是当前时间步的输出。$f_W$ 通常是一个带有非线性激活函数的线性变换，用于更新隐藏状态。$f_y$ 用于把隐藏状态转成输出维度，同时也是一个权重矩阵，负责将隐藏状态映射到输出空间。

在处理向量序列时，每个时间步都会套用 $h_t = f_W(h_{t-1}, x_t)$ 这个公式来计算隐藏状态，但在预测输出时，公式往往与计算隐藏状态的公式不同。

在经典的 vanilla RNN 中，$f_W$ 通常使用 tanh 作为激活函数，以防止梯度在反向传播过程中过快爆炸或消失。其展开形式为：

$$
h_t = \tanh(W_{hh} h_{t-1} + W_{xh} x_t)
$$

##### Many to Many 任务

RNN 可以用于多种类型的任务，包括 many to one、one to many 以及 many to many。其中 many to many 任务要求模型在每一个时间步都产生一个输出，例如视频的逐帧标注或词性标注。

![RNN实例](../img/RNN_exa.png)

在上图所示的例子中，任务被定义为一个 many to many 的序列建模问题，目标是从输入的序列中检测连续的 1。例如，输入序列为 $0, 1, 0, 1, 1, 1, 0, 1, 1$，模型需要输出对应位置的预测，判断当前或之前是否出现了连续的两个 1。

---


In [ ]:
import numpy as np

# 定义 ReLU 激活函数
def relu(x):
    return np.maximum(0, x)

# 初始化权重矩阵，数值严格按照图中给定
# w_xh: 输入到隐藏状态的权重，形状 (3, 1)
w_xh = np.array([[1], 
                 [0], 
                 [0]])

# w_hh: 隐藏状态到隐藏状态的循环权重，形状 (3, 3)
# 作用是将上一时刻的“当前值”转移到“前一个值”位置，保留常数项 1
w_hh = np.array([[0, 0, 0], 
                 [1, 0, 0], 
                 [0, 0, 1]])

# w_hy: 隐藏状态到输出的权重，形状 (1, 3)
# 权重设置为 [1, 1, -1]，用于计算当前值 + 上一个值 - 1
w_hy = np.array([[1, 1, -1]])

# 输入序列 X，目标是检测连续的 1
x_seq = [0, 1, 0, 1, 1, 1, 0, 1, 1]

# 初始隐藏状态 h_0，形状 (3, 1)
# 包含三个分量：[当前值, 前一个值, 常数项1]
h_t_prev = np.array([[0], 
                     [0], 
                     [1]])

print("X\tRNN\tY")
print("-" * 20)

# 记录每个时间步的输出结果
y_seq = []

for t, x in enumerate(x_seq):
    # 1. 计算当前隐藏状态
    # h_t = ReLU(W_hh * h_{t-1} + W_xh * x_t)
    # 注意：这里的 @ 表示矩阵乘法
    h_t = relu(w_hh @ h_t_prev + w_xh * x)
    
    # 2. 计算当前输出
    # y_t = ReLU(W_hy * h_t)
    y_t = relu(w_hy @ h_t)
    
    # 3. 更新上一时刻的隐藏状态，用于下一次迭代
    h_t_prev = h_t
    
    # 收集输出结果
    y_seq.append(y_t.item())
    
    # 打印当前的输入和输出，展示对应关系
    print(f"{x}\t->\t{int(y_t.item())}")

print("-" * 20)
print("最终预测输出 Y:", y_seq)


##### Vanilla RNN 实例

在具体的代码实现中，这个检测连续 1 的任务被巧妙地通过矩阵运算实现了。代码中定义了三组权重矩阵，它们各自承担了非常明确的功能。

```python
w_xh = np.array([[1], [0], [0]])
w_hh = np.array([[0, 0, 0], [1, 0, 0], [0, 0, 1]])
w_hy = np.array([[1, 1, -1]])
```

**隐藏状态 $h_t$ 的三个分量**

在具体解析之前，需要先明确隐藏状态 $h_t$ 的三个分量所代表的物理意义。根据图中的注释，隐藏状态 $h_t$ 是一个三维向量，其三个分量分别代表：

- 分量1：当前输入的值 Current
- 分量2：上一个输入的值 Previous
- 分量3：常数项 1，用于提供偏置 Bias

**权重矩阵的具体工作**

1. **输入权重矩阵 `w_xh`**

   `w_xh` 的形状是 $3 \times 1$。它的作用是把输入标量 $x_t$ 映射到隐藏状态向量的三个维度上。
   根据矩阵乘法，$[3 \times 1] \times x_t = [3 \times 1]$，结果相当于把 $x_t$ 放到第一个分量，第二个和第三个分量置为 0。这正好对应了“将当前值记录到隐藏状态的第一个分量中”这一逻辑。

2. **循环权重矩阵 `w_hh`**

   `w_hh` 的形状是 $3 \times 3$。它的作用是将上一时刻的隐藏状态 $h_{t-1}$ 进行线性组合，形成当前时刻隐藏状态的初始部分。
   通过矩阵乘法，$w_{hh} \times h_{t-1}$ 完成的是：把上一时刻的“当前值”复制到当前时刻的“上一个值”位置，同时保留常数项 1，并重置“当前值”为 0。
   这个操作实现了状态的转移：当前时刻的“历史”变成了上一时刻的“实时”。

3. **输出权重矩阵 `w_hy`**

   `w_hy` 的形状是 $1 \times 3$。它的作用是从隐藏状态中提取信息，计算最终输出。
   它的权重是 $[1, 1, -1]$。结合隐藏状态 $h_t$ 的三个分量，输出计算为：$1 \times \text{Current} + 1 \times \text{Previous} - 1 \times 1$。
   具体逻辑是：当且仅当“当前值”和“上一个值”都为 1 时，计算结果为 $1+1-1=1$。否则，如果只有一个 1，结果为 $1+0-1=0$；如果没有 1，结果为 $0+0-1=-1$。

**前向传播与激活函数**

```python
h_t_prev = np.array([[0], [0], [1]])
for t, x in enumerate(x_seq):
    h_t = relu(w_hh @ h_t_prev + w_xh @ x)
    y_t = relu(w_hy @ h_t)
    h_t_prev = h_t
```

循环中的代码使用了 ReLU 作为激活函数。对于 $h_t$，由于 $w_{hh} @ h_{t-1} + w_{xh} @ x_t$ 产生的结果中，前两个分量通常大于等于 0，而第三个分量恒为 1，ReLU 实际上起到了一个取最大值的作用。在这种情况下，$h_t$ 的三个分量始终保持非负。

对于 $y_t$，由于 $w_{hy} @ h_t$ 计算结果可能为 $-1$、$0$ 或 $1$。当输出为 $1$ 时，经过 ReLU 后依然为 $1$；当输出为 $0$ 时，经过 ReLU 后依然为 $0$；当输出为 $-1$ 时，经过 ReLU 后变成 $0$。

因此，这个具体的 RNN 例子实际上是通过手工设计权重，实现了在任意序列上检测是否出现了连续两个 1 的功能。这也是 RNN 前向传播的一个很直观的示例：通过矩阵运算和激活函数，让隐藏状态不断更新，从而提取序列中的时序依赖关系。



#### RNN 的损失与梯度计算

RNN 的训练过程与普通的全连接网络类似，但因为涉及到时间维度，其前向传播和反向传播都必须在序列上逐步展开。

##### 前向传播与损失计算

以 Andrej Karpathy 的极简字符级 RNN 代码 `min-char-rnn.py` 为例，模型在每一个时间步接收一个字符输入，输出下一个字符的预测概率分布。

前向传播的核心流程如下：

1. 计算当前隐藏状态：$h = \tanh(W_{xh}x + W_{hh}h_{prev} + b_h)$
2. 计算未归一化的输出分数：$y = W_{hy}h + b_y$
3. 通过 softmax 得到概率分布：$p = \exp(y) / \sum \exp(y)$
4. 根据目标字符计算交叉熵损失，并累加到总损失中。

**随时间反向传播**

反向传播过程中，误差需要从序列的末端逐级向前传递，这被称为随时间反向传播 Backpropagation Through Time, BPTT。

序列中的每一个时间步都会对最终的损失函数有贡献。因此，损失函数对每个时间步的隐藏状态的梯度都需要累加。
```python
dh = np.dot(Why.T, dy) + dh_next # 结合当前步和未来步的梯度
dhraw = (1 - h * h) * dh          # 经过 tanh 的反向传播
dbh += dhraw                      # 累加偏置梯度
dWxh += np.dot(dhraw, xs[t].T)    # 累加输入权重梯度
dWhh += np.dot(dhraw, hprev.T)    # 累加循环权重梯度
```
在计算 `dh` 时，`dh_next` 是未来时间步传回来的梯度，这一步实现了梯度在时间上的反向流动。

因为参数 $W_{xh}$、$W_{hh}$、$W_{hy}$ 在时间维度上是共享的，所以它们在整个序列上的梯度是各个时间步梯度累加的结果。这就是笔记中提到的“梯度相加”。

根据任务需求，有时只需要最后一个时间步的梯度用于分类，有时则需要每个时间步的隐藏状态梯度（例如视频分类中的逐帧标注）。因此，需要将每个时间步的激活值和梯度都储存下来，但这会导致长序列的内存开销急剧增大。

**截断随时间反向传播**

为了解决长序列内存开销过大的问题，引入了截断随时间反向传播 Truncated BPTT。

它的做法是将长序列切分成若干个固定长度的子序列，比如每 50 个时间步作为一个批次。前向传播和反向传播只在子序列内部进行，批次与批次之间不再传递梯度。这使得计算图的大小被固定，内存开销变得可控，但代价是模型无法捕获超过这个截断长度的长期依赖关系。

**梯度裁剪**

在 BPTT 过程中，梯度可能会因为链式法则的累积而爆炸。因此，通常需要加入梯度裁剪，将梯度的绝对值限制在某个范围内，例如 5 以内。
```python
for dparam in [dWxh, dWhh, dWhy, dbh, dby]:
    np.clip(dparam, -5, 5, out=dparam)
```

这个步骤能够有效防止梯度爆炸，保证训练的稳定性。

**从模型里采样**

在生成文本时，模型并不总是选择概率最高的字符。如果总是选择概率最高的字符，生成的文本会非常单一且缺乏创造性。因此，通常采用随机采样，根据输出的概率分布进行抽样，这样可以在保证合理性的同时，增加生成结果的多样性。

**搜索可解释单元**

训练好的 RNN 模型内部可能存在一些有特定功能的神经元，比如行长追踪单元，它们能够追踪文本中引号的开闭状态或代码块的缩进层级。通过搜索这些可解释单元，可以帮助理解模型究竟在多大程度上学到了符号级的规律。

#### RNN 的优缺点与应用

**优点**

- 理论上，RNN 没有上下文长度的限制，它可以处理任意长度的序列。
- 模型的大小不会随时间步的增加而增大，因为所有权重在时间步上都是共享的。
- 这使得 RNN 非常适合处理自然语言和音频等变长序列。

**缺点**

- RNN 采用递归计算，导致计算过程无法并行化，耗时较长。
- 实际上 RNN 很难获取很多时间步之前的信息，因为梯度在反向传播过程中容易消失或爆炸，导致长距离依赖难以学习。

##### RNN 的应用

在计算机视觉领域，RNN 也有广泛的应用：

- 图像描述生成：模型输入一张图片，输出一段描述该图片的文字。但这种应用存在幻觉问题，模型可能建立了某种连接，但并没有真正建立可解释的因果理解。
- 视觉问答：模型需要根据图像内容回答自然语言问题。
- 视觉导航任务：模型需要根据视觉输入做出连续的路径决策。

#### 多层 RNN 与 LSTM

**多层 RNNs**

多层 RNN 通过在深度方向叠加多个 RNN 层，让第一层的输出作为第二层的输入，以此类推。这增加了模型的容量，使得网络能够学习到更高级的抽象特征。

**LSTM**

RNN 有先天不足，容易发生梯度消失或梯度爆炸，导致信息捕获不足。为了解决这个问题，引入了长短期记忆网络 LSTM。

LSTM 引入了门控机制和细胞状态。其核心机制可以拆解为以下几个部分：

- 细胞状态：相当于一条信息高速公路，梯度可以沿着它几乎无衰减地传播。
- 遗忘门：决定丢弃哪些旧信息。
- 输入门：决定写入哪些新信息。
- 输出门：决定输出哪些信息。

这使得 LSTM 能够在长序列中有效地捕获并保留长期依赖关系，成为 RNN 家族中最具代表性的改进架构。


In [ ]:
import numpy as np

# 1. 数据准备
# 演示用文本数据，直接内置避免依赖外部文件
data = "hello world. this is a simple char rnn example. " * 20
chars = list(set(data)) # 获得所有不重复字符
data_size, vocab_size = len(data), len(chars)
print(f"数据长度: {data_size}, 词汇表大小: {vocab_size}")

# 字符与整数索引的互相映射
char_to_ix = {ch: i for i, ch in enumerate(chars)}
ix_to_char = {i: ch for i, ch in enumerate(chars)}

# 2. 超参数与初始化
hidden_size = 100    # 隐藏层神经元数量
seq_length = 25      # 每个批次的序列长度 (BPTT 的截断长度)
learning_rate = 1e-1 # 学习率

# 模型参数初始化 (使用小方差随机初始化，防止初始饱和)
Wxh = np.random.randn(hidden_size, vocab_size) * 0.01  # 输入到隐藏层的权重
Whh = np.random.randn(hidden_size, hidden_size) * 0.01 # 隐藏层循环权重
Why = np.random.randn(vocab_size, hidden_size) * 0.01  # 隐藏层到输出层的权重
bh = np.zeros((hidden_size, 1))                        # 隐藏层偏置
by = np.zeros((vocab_size, 1))                         # 输出层偏置

# Adagrad 优化器内存变量 (累积梯度平方，实现自适应学习率)
mWxh, mWhh, mWhy = np.zeros_like(Wxh), np.zeros_like(Whh), np.zeros_like(Why)
mbh, mby = np.zeros_like(bh), np.zeros_like(by)

# 3. 前向与反向传播
def lossFun(inputs, targets, hprev):
    """
    前向计算损失，反向计算梯度 (BPTT)。
    inputs/targets: 字符索引列表
    hprev: 上一批次的最终隐藏状态 (用于批次间传递)
    """
    xs, hs, ys, ps = {}, {}, {}, {}
    hs[-1] = np.copy(hprev) # 初始化前一步隐藏状态
    loss = 0
    
    # ---- 前向传播 ----
    for t in range(len(inputs)):
        xs[t] = np.zeros((vocab_size, 1)) # one-hot 编码
        xs[t][inputs[t]] = 1
        
        # 计算当前隐藏状态：结合当前输入和上一隐藏状态
        hs[t] = np.tanh(np.dot(Wxh, xs[t]) + np.dot(Whh, hs[t-1]) + bh)
        
        # 计算未归一化得分 logits 和 softmax 概率
        ys[t] = np.dot(Why, hs[t]) + by
        ps[t] = np.exp(ys[t]) / np.sum(np.exp(ys[t]))
        
        # 累加交叉熵损失
        loss += -np.log(ps[t][targets[t], 0])
        
    # ---- 反向传播 (BPTT) ----
    dWxh, dWhh, dWhy = np.zeros_like(Wxh), np.zeros_like(Whh), np.zeros_like(Why)
    dbh, dby = np.zeros_like(bh), np.zeros_like(by)
    dh_next = np.zeros_like(hs[0]) # 未来时间步传回的梯度
    
    for t in reversed(range(len(inputs))):
        # softmax + 交叉熵的反向传播
        dy = np.copy(ps[t])
        dy[targets[t]] -= 1 # 目标类概率减1，其余不变
        
        dWhy += np.dot(dy, hs[t].T)
        dby += dy
        
        # 梯度相加：当前步输出贡献 + 未来步传回
        dh = np.dot(Why.T, dy) + dh_next
        
        # 反向传播通过 tanh 激活函数
        dhraw = (1 - hs[t] * hs[t]) * dh 
        
        dbh += dhraw
        dWxh += np.dot(dhraw, xs[t].T)
        dWhh += np.dot(dhraw, hs[t-1].T)
        
        # 将梯度继续向前一时间步传递
        dh_next = np.dot(Whh.T, dhraw)
        
    # 梯度裁剪，防止梯度爆炸
    for dparam in [dWxh, dWhh, dWhy, dbh, dby]:
        np.clip(dparam, -5, 5, out=dparam)
        
    return loss, dWxh, dWhh, dWhy, dbh, dby, hs[len(inputs)-1]

# 4. 采样生成
def sample(h, seed_ix, n):
    """根据当前模型和隐藏状态，生成指定长度的字符序列"""
    x = np.zeros((vocab_size, 1))
    x[seed_ix] = 1 # 种子字符
    ixes = []
    
    for t in range(n):
        h = np.tanh(np.dot(Wxh, x) + np.dot(Whh, h) + bh)
        y = np.dot(Why, h) + by
        p = np.exp(y) / np.sum(np.exp(y))
        
        # 按概率随机采样，而不是每次取最大值
        ix = np.random.choice(range(vocab_size), p=p.ravel()) 
        x = np.zeros((vocab_size, 1))
        x[ix] = 1
        ixes.append(ix)
        
    return ixes

# 5. 主训练循环
n, p = 0, 0
# 平滑损失，用于过滤单次波动，观察整体趋势
smooth_loss = -np.log(1.0 / vocab_size) * seq_length 

while True:
    # 当数据指针走到末尾，或者刚开始时，重置指针和隐藏状态
    if p + seq_length + 1 >= len(data) or n == 0:
        hprev = np.zeros((hidden_size, 1))
        p = 0
        
    # 准备输入和目标序列 (目标向右偏移一位)
    inputs = [char_to_ix[ch] for ch in data[p:p+seq_length]]
    targets = [char_to_ix[ch] for ch in data[p+1:p+seq_length+1]]
    
    # 每 100 步采样一次，观察生成效果
    if n % 100 == 0:
        sample_ix = sample(hprev, inputs[0], 200)
        txt = ''.join(ix_to_char[ix] for ix in sample_ix)
        print(f"----\n {txt} \n----")
        
    # 前向与反向传播
    loss, dWxh, dWhh, dWhy, dbh, dby, hprev = lossFun(inputs, targets, hprev)
    smooth_loss = smooth_loss * 0.999 + loss * 0.001
    
    if n % 100 == 0:
        print(f"迭代次数: {n}, 平滑损失: {smooth_loss}")
        
    # Adagrad 参数更新 (自适应调整每个参数的学习率)
    for param, dparam, mem in zip([Wxh, Whh, Why, bh, by],
                                  [dWxh, dWhh, dWhy, dbh, dby],
                                  [mWxh, mWhh, mWhy, mbh, mby]):
        mem += dparam * dparam
        param += -learning_rate * dparam / np.sqrt(mem + 1e-8)
        
    p += seq_length
    n += 1

---


传统的 Seq2Seq 模型由编码器和解码器组成。编码器是一个 RNN，负责把输入序列（比如一句话）压缩成一个固定长度的最终隐藏状态，作为解码器的初始状态 $s_0$。解码器是另一个 RNN，根据 $s_0$ 和已生成的词，逐个生成翻译结果。

但这种“固定长度压缩”存在瓶颈。若输入序列很长，早期的信息会在传递过程中严重丢失。

我们希望生成输出时能让网络能回顾所有输入序列。

注意力机制就是为了解决这个问题，它让解码器在生成每一个词时，都能直接“回看”编码器所有的隐藏状态，动态选择最相关的部分。

#### RNN and Attention
![Seq2Seq with Attention](../img/Attention.png)

如上图所示，左侧是编码器（蓝色方块 $h_1, h_2, h_3, h_4$），输入是 “we see the sky”。右侧是解码器（浅蓝色方块 $s_0, s_1, s_2$），生成翻译 “vediamo il”。

注意力机制的核心直觉是：**上下文向量（Context Vector）会去关注输入序列中相关的部分。**

比如，当解码器要生成 “vediamo”（对应 “we see”）时，它会重点关注输入中的 “we” 和 “see”，给它们分配更高的注意力权重，比如 $a_{11}=a_{12}=0.45$，而给 “the” 和 “sky” 分配较低的权重 $a_{13}=a_{14}=0.05$。当生成 “il”（对应 “the”）时，注意力权重 $a_{24}$ 会高达 $0.8$，集中在 “the” 这个词上。



注意力机制的计算过程是**完全可微的**，不需要对注意力权重进行额外的监督，梯度会通过反向传播自动更新。

计算步骤可以分为以下四步：

1. **计算对齐分数 Attention Scores**：
   将解码器上一时刻的隐藏状态 $s_{t-1}$ 与编码器每个时间步的隐藏状态 $h_i$ 进行对比，计算一个标量对齐分数 $e_{t,i}$。
   $$e_{t,i} = f_{att}(s_{t-1}, h_i)$$
   其中 $f_{att}$ 是一个线性层（Linear Layer），把两个向量映射成一个分数。

2. **归一化得到注意力权重**：
   使用 Softmax 将所有的对齐分数转化为概率分布，得到注意力权重 $a_{t,i}$。
   $$a_{t,i} = \frac{\exp(e_{t,i})}{\sum_j \exp(e_{t,j})}$$
   这保证了所有 $a_{t,i} \in (0, 1)$，且 $\sum_i a_{t,i} = 1$。

3. **计算上下文向量 Context Vector**：
   使用注意力权重对编码器的隐藏状态进行加权求和，得到当前时间步的上下文向量 $c_t$。
   $$c_t = \sum_i a_{t,i} h_i$$
   这个上下文向量就是输入序列的“动态浓缩版”，它聚焦于当前生成步骤最需要的输入信息。

4. **更新解码器状态**：
   将当前时间步的上下文向量和上一时刻生成的词结合，更新解码器当前的隐藏状态。
   $$s_t = g_U(y_{t-1}, s_{t-1}, c_t)$$
   其中 $y_{t-1}$ 是上一时刻生成的词，比如 $y_0$ 是 `[START]`，$y_1$ 是 `vediamo`。$g_U$ 是解码器的循环单元（比如 LSTM 或 RNN）。
   
   每个时间步的上下文向量 $c_t$ 是**不同**的。编码器的输入序列不经过单个固定向量的瓶颈压缩，而是在解码器的每一个时间步，上下文向量 $c_t$ 都会动态地“看向”输入序列的不同部分。例如在图中，生成 $y_1$（vediamo）时，$c_1$ 重点关注 "we" 和 "see"；生成 $y_2$（il）时，$c_2$ 重点关注 "the"。这让解码器在每一步都能获取最需要的信息，完美解决了长序列的信息瓶颈问题。

![Seq2Seq with Attention2](../img/Attention2.png)

- **初始解码器状态 $s_0$**：一般地，通常将编码器最后一个时间步的隐藏状态作为解码器的初始状态 $s_0$。
- **动态注意力映射**：中间部分的 $e_{21}$ 到 $e_{24}$，$a_{21}$ 到 $a_{24}$ 展示了在解码第 2 个词时的对齐与权重。图示中 $s_0$ 参与计算，并与输入序列混合。
- **下一个隐藏状态**：图解中缺少了 $s_0$ 指向 $s_1$ 的箭头。解码器当前的隐藏状态 $s_1$ 必须接收上一时刻的隐藏状态 $s_0$。在 Seq2Seq 中，$s_0$ 通常被初始化为编码器最后一个时间步的隐藏状态 $h_4$（或全零向量），它作为解码器的初始记忆随序列推进。
- **反向传播式监督**：整个注意力机制“**All differentiable! No supervision on attention weights. Backprop through everything.**” 意思是：注意力权重完全是通过反向传播学习出来的。模型自己在训练过程中发现应该把注意力放在哪个词上，不需要人工去告诉它“这句话里的 the 翻译成 il”，它会自己学会将 “the” 的 $a$ 值调高来优化翻译结果。
---

####  Attention Layer

我们把上述机制泛化一下。
注意力机制的本质可以提炼为一个通用的算子。它不关心查询向量从哪来，只负责计算查询向量与数据向量之间的相似度，进而得出输出向量。这使得注意力机制能够作为一个独立的模块（Lego block），被灵活嵌入到各种网络架构中。


在通用的注意力层中，主要有以下几组输入：

- **查询向量 Query**：$Q$，形状为 $[N_Q \times D_Q]$。$N_Q$ 是查询的数量，$D_Q$ 是查询的维度。它代表当前想要寻找的信息。
- **数据向量 Data Vectors**：$X$，形状为 $[N_X \times D_Q]$。$N_X$ 是数据向量的数量，$D_Q$ 同样是对齐的维度。它代表可供查询的原始信息库。
- **输出向量 Output**：$Y$，形状为 $[N_Q \times D_X]$。对于每一个查询向量，都会得到一个对应的输出向量。这个输出向量是**数据向量的线性组合**。


![AL_](../img/AttentionLayer_.png)
注意力机制的计算分为三步：计算相似度分数、归一化得到注意力权重、加权求和得到输出。

**1.计算相似度分数矩阵**

为了让查询向量与数据向量进行交互，最直接的方式是计算它们之间的相似度。这里使用的是**点积**。

$$
E = \frac{Q X^T}{\sqrt{D_Q}}
$$

矩阵乘法的维度要求体现了**列对齐**。$Q$ 的每一行代表一个查询向量，$X$ 的每一行代表一个数据向量。由于它们都有 $D_Q$ 维度，$Q$ 乘以 $X$ 的转置后，会得到一个形状为 $[N_Q \times N_X]$ 的**相似度分数矩阵** $E$。$E$ 的每一个元素 $E_{ij}$ 就是一个标量，代表第 $i$ 个查询与第 $j$ 个数据向量的相似度。

**为什么除以 $\sqrt{D_Q}$？**

点积的结果会随着维度 $D_Q$ 的增加而变大，这会导致 softmax 的输入数值过大，输出分布极其尖锐，反向传播时梯度会非常小（类似梯度消失）。因此，除以 $\sqrt{D_Q}$ 是为了**缩放点积**，保持方差不随维度增长，防止梯度消失，让训练更稳定。

**2.计算注意力权重**

对相似度分数矩阵在特定维度上做 softmax，将其转化为概率分布。

$$
A = \text{softmax}(E, \text{dim}=1)
$$

这里的 $\text{dim}=1$ 指的是在 $N_X$ 这个维度（即列的方向）进行归一化。这意味着，对于每一个查询 $Q_i$，它对应的注意力权重 $A_{i,:}$ 在所有数据向量上的总和为 1。

**3.计算输出向量**

用注意力权重对数据向量进行加权求和，得到输出向量。

$$
Y = A V
$$

这个矩阵乘法实现了**用另一个矩阵的值做权重，对输入矩阵的列做线性组合**。$A$ 的每一行是对应查询的权重分配，$V$ 的每一行是数据向量的值。最终 $Y$ 的第 $i$ 行，就是所有数据向量的线性组合，权重由 $A$ 的第 $i$ 行决定。

##### 引入键和值（Key-Value）的概念

在更灵活的设计中，我们不再直接使用原始的数据向量 $X$，而是将 $X$ 分别映射成**键（Key）**和**值（Value）**。

- **键矩阵** $W_K$：$[D_x \times D_Q]$
- **值矩阵** $W_V$：$[D_x \times D_V]$

$$
K = X W_K
$$

$$
V = X W_V
$$

这样做的目的是让注意力算子独立。原始数据 $X$ 可能包含复杂的信息，通过引入 $W_K$ 和 $W_V$，网络可以学习到“应该用什么特征去匹配查询”（键），以及“匹配成功后应该传递什么信息”（值）。此时，相似度计算变为：

$$
E = \frac{Q K^T}{\sqrt{D_Q}}
$$

输出向量变为：

$$
Y = A V
$$

![AL](../img/AttentionLayer.png)

**多查询与多维矩阵运算**

实际应用中，我们往往**一次使用多个查询向量**。这意味着 $Q$ 是一个矩阵（$N_Q$ 行）。矩阵运算的优势在此刻展现得淋漓尽致：

- 相似度分数 $E$ 的计算，本质上是在做列对齐的批量点积。
- Softmax 操作在维度 $N_X$ 上进行，保证了每个查询向量都有自己专属的注意力分布。
- 最终输出 $Y$ 的每一行，都对应一个查询向量的输出结果。

右侧的图解生动地展示了这个过程。$Q_1, Q_2, Q_3, Q_4$ 分别与 $K_1, K_2, K_3$ 计算相似度分数 $E_{ij}$。经过 softmax 得到 $A_{ij}$ 后，再与 $V_1, V_2, V_3$ 进行加权求和（Product 和 Sum），最终得到 $Y_1, Y_2, Y_3, Y_4$。

整个过程中，注意力算子完全由矩阵乘法和 softmax 组成，**所有操作都是可微的，并且没有引入额外的监督信号**。这意味着反向传播可以自动优化 $W_K$、$W_V$ 以及查询向量的生成逻辑，让模型自己学会在哪里“看”，以及看什么。


#### 自注意力层 Self-Attention Layer

自注意力机制的本质，是将输入向量序列本身作为查询、键和值的来源，通过计算序列内部各个元素之间的关联程度，来生成新的特征表示。这在现代架构（如 Transformer）中是最核心的组件。



自注意力层在计算时，首先将输入向量投影成三部分：查询 $Q$、键 $K$ 和值 $V$。这三部分由三个可学习的权重矩阵生成。

- **输入向量**：$X$，形状为 $[N \times D]$。$N$ 是序列长度，$D$ 是输入特征维度。
- **查询矩阵**：$W_Q$，形状为 $[D \times D_H]$。
- **键矩阵**：$W_K$，形状为 $[D \times D_H]$。
- **值矩阵**：$W_V$，形状为 $[D \times D_H]$。
- **输出矩阵**：$W_O$，形状为 $[D_H \times D]$。

这里 $D$ 和 $D_H$ 都是超参数，一般在模型设计时指定。$D_H$ 通常等于 $D / H$（$H$ 为注意力头数）。

通过线性投影，我们得到：
$$
Q = X W_Q
$$
$$
K = X W_K
$$
$$
V = X W_V
$$


**如何选取自注意力与交叉注意力？**

- **自注意力（Self-Attention）**：$Q, K, V$ 全部来源于同一个输入序列。这通常用于编码器内部，或者解码器内部，让序列自己跟自己进行交互，提取上下文关系。
- **交叉注意力（Cross-Attention）**：$Q$ 来自一个序列（比如解码器），而 $K$ 和 $V$ 来自另一个序列（比如编码器的输出）。这在机器翻译等任务中非常常见，让解码器能够在生成翻译时，去关注编码器理解的源语言信息。

**置换等变性（Permutation Equivariance）**


既然自注意力是计算序列内部元素的关系，那么如果交换输入序列的顺序，会发生什么？

**输出序列也会发生相同顺序的交换，但每个位置的输出内容不会改变。** 这就是置换等变性。

这也意味着，**自注意力机制本身不在乎输入的顺序**。它处理的是一个“向量集合”，而不是一个序列。因此，如果在处理文本或时间序列等对顺序敏感的数据时，必须额外引入**位置嵌入（Positional Encoding）**，将位置信息注入到输入向量中，这样自注意力才能区分不同位置的元素。

**Masked Self-Attention Layer（掩码自注意力）**

在仅解码器架构（如 GPT）中，模型是自回归生成的，即预测下一个词时只能看到前面的词，不能提前看到后面的词。

如果我们使用标准的自注意力，模型会“偷看”后面的词，导致训练无效。因此需要引入掩码。

具体做法是在计算相似度分数矩阵 $E$ 之后，进行 softmax 之前，将未来位置的分数（即矩阵上三角部分）设置为负无穷（$-\infty$）。这样 softmax 之后，这些位置的注意力权重就变成了 0，模型只能关注到当前及之前的位置。

**Multi-headed（多头注意力）**

$H$ 代表注意力头的数量。通过将 $D$ 维度拆分成 $H$ 个 $D_H$ 维度（$D_H = D / H$），模型可以在不同的子空间中独立地执行注意力计算。

这就像让多个专家同时阅读同一段文本，每个人关注不同的特征（比如有的关注语法，有的关注语义），最后将所有人的结论拼接起来，从而增强了模型的表达能力。这也是为什么自注意力层的计算中会出现 $H$ 这个维度。

考虑多头注意力（Multi-headed），$Q, K, V$ 的形状会包含 $H$ 维度，即 $[H \times N \times D_H]$。

**核心计算就是四个矩阵乘法**

![M](../img/SelfAttention.png)
自注意力层的核心，实际上是四个矩阵乘法完成的。我们可以按以下步骤拆解：


1. **QKV 投影**：
   将输入投影到查询、键和值空间。
   $$[N \times D] \times [D \times 3 H D_H] \implies [N \times 3 H D_H]$$
   实际操作中，$W_Q, W_K, W_V$ 可以合并成一个大的权重矩阵 $W_{QKV}$，一次性计算完成后，再拆分并重塑得到 $Q, K, V$ 三个张量，每个形状为 $[H \times N \times D_H]$。

2. **QK 相似度**：
   计算查询和键的点积，得到相似度分数。
   $$[H \times N \times D_H] \times [H \times N \times D_H]^T \implies [H \times N \times N]$$
   计算出的结果 $E = Q K^T / \sqrt{D_Q}$，代表第 $i$ 个位置对第 $j$ 个位置的关注程度（分数）。

3. **V 加权**：
   将相似度分数进行 softmax 归一化得到注意力权重 $A$，然后用它们对值进行加权求和。
   $$[H \times N \times N] \times [H \times N \times D_H] \implies [H \times N \times D_H]$$
   （注：幻灯片右侧中此处的维度写成了 $[H \times D \times D_H]$，这属于笔误，正确的是以 $V$ 的实际形状 $[H \times N \times D_H]$ 参与计算）
   计算得到 $Y = A V$，然后将其重塑为 $[N \times H D_H]$，把多个头的输出拼接在一起。

4. **输出投影**：
   将拼接后的多头输出映射回原始的维度。
   $$[N \times H D_H] \times [H D_H \times D] \implies [N \times D]$$
   即 $O = Y W_O$，最终输出维度与输入维度一致。

---

![](../img/three.png)

序列数据如文本、音频和时间序列在深度学习中有三种主要的处理范式。这三者在计算模式、并行化能力以及长序列处理上各有优劣。

**循环神经网络 Recurrent Neural Network**

循环神经网络按照时间步顺序处理数据。如图所示，$y_1 \rightarrow y_2 \rightarrow y_3 \rightarrow y_4$ 之间存在着明确的顺序依赖。每个时间步的输出 $y_t$，不仅依赖于当前的输入 $x_t$，还依赖于上一个时间步的输出 $y_{t-1}$，或者说隐藏状态。这种机制使它天然适合处理一维有序序列。

- **优点**：理论上对长序列有效。对于长度为 $N$ 的序列，计算和内存复杂度呈线性增长，即 $O(N)$。

- **缺点**：无法并行化。因为当前时间步的隐藏状态必须等待上一个时间步计算完成，只能顺序计算，这在现代硬件上会极大限制训练速度。

**卷积 Convolution**

卷积网络通过滑动窗口来处理序列。图中展示了 $y_2$ 是由 $x_1$ 和 $x_2$ 共同计算得出的，每个输出只依赖于输入的一个局部窗口。这种方法天然适合处理 N 维网格结构。

- **优点**：高度并行化。因为卷积核在序列上滑动时，每个输出之间的计算相互独立，可以同时并行计算。

- **缺点**：不适合长序列，因为每个输出只能看到局部输入。为了扩大感受野，必须堆叠很多层卷积，这会导致计算开销急剧增加。

**自注意力 Self-Attention**

自注意力机制打破了顺序和局部的限制。图中清晰地展示了，每个输出 $y_1, y_2, y_3, y_4$ 都直接与所有的输入 $x_1, x_2, x_3, x_4$ 计算注意力关联。它把输入看作是一个向量集合，而不是有序序列。

- **优点**：适合长序列，因为每个输出都直接依赖于所有输入，不存在信息传递的瓶颈。同时它高度并行化，本质上只是四个矩阵乘法。

- **缺点**：计算昂贵。对于长度为 $N$ 的序列，计算复杂度是 $O(N^2)$，因为每个输入都要和所有输入做相似度计算，而内存复杂度是 $O(N)$。


循环神经网络强调顺序依赖，但牺牲了并行性。卷积强调局部特征和并行性，但需要堆叠层数来扩大视野。自注意力则放弃了局部和顺序的先验，换取全局视野和极致的并行化，但带来了平方级的计算开销。

在现代架构设计中，自注意力机制因算力提升和模型规模扩大而逐渐成为主流，但通常也需要结合位置嵌入来弥补其不关心顺序的缺陷。三种方式的选择往往取决于具体的任务需求、序列长度以及可用的计算资源。

#### The Transformer

Transformer 架构出自 2017 年的经典论文《Attention is all you need》。它的核心设计是一个 **Transformer Block**（Transformer 模块），输入是一组向量集合 $x$，输出也是一组向量集合 $y$。整个模块的设计高度模块化，可以不断堆叠。

![Transformer](../img/Transformer.png)

**数据流动与组件**

从架构图的最底部开始，数据依次经过以下阶段：

1. **输入向量 $x_1, x_2, x_3, x_4$**：这是序列中每个元素的向量表示。
2. **Self-Attention 层（多头自注意力层）**：输入首先进入这里。这是整个模块中**向量之间唯一的交互点**。“All vectors interact through (multiheaded) Self-Attention”，只有在此时，向量才会去“看”序列中的其他向量，并根据相关性（注意力权重）融合信息。这也是“多头”概念的体现，每个头在不同的子空间独立计算注意力，最后拼接。
3. **残差连接（$\oplus$）**：自注意力的输出会与进入自注意力之前的原始输入相加。这被称为残差连接，它是**训练深层网络的关键**。它让梯度可以直接通过这条“高速公路”回传，有效缓解了梯度消失问题。
4. **Layer Normalization（层归一化）**：残差相加后的结果进入 LayerNorm。“Layer normalization normalizes all vectors”，它负责将每个样本的特征维度调整为均值为0、方差为1的分布，**加速收敛并稳定训练**。
5. **MLP 层（多层感知机）**：数据接着进入标为 MLP 的四个独立方块。“MLP independently on each vector”。这意味着，除了前面的自注意力层，这里的 MLP 和 LayerNorm 都对每个向量**独立处理**。它们只在特征维度上进行变换，向量之间不交换任何信息。
6. **第二次残差连接与 LayerNorm**：MLP 的输出再次加上进入 MLP 之前的输入，然后经过第二个归一化层。
7. **输出向量 $y_1, y_2, y_3, y_4$**：最终得到与输入数量相同、但包含了丰富上下文信息的输出向量集合。

**核心概念与原理**

**6 个矩阵乘法**

Transformer 高度可扩展和可并行化，核心计算量主要来源于 **6 个矩阵乘法（Matmul）**。

- **4 个来自 Self-Attention**：分别是（1）$Q, K, V$ 的投影；（2）$QK^T$ 计算相似度；（3）$AV$ 计算加权输出；（4）输出投影 $YW_O$。
- **2 个来自 MLP**：MLP 通常由两个线性层组成，中间夹着激活函数，因此对应两个矩阵乘法。

这种“纯矩阵乘法”的结构，极大契合了 GPU 和 TPU 的并行计算架构，使得 Transformer 能够轻松扩展到千亿、万亿参数。

**高度可并行化**

在传统的 RNN 中，计算必须按时间步顺序进行，前面的算不完，后面的就得等。而在 Transformer 中，由于 Self-Attention 可以通过矩阵乘法一次性算出所有位置的注意力，**序列中所有位置的运算完全可以同时进行**。此外，MLP 和 LayerNorm 本身就是逐向量独立操作，天然适合并行。

**输入是集合（Set of vectors），输出是集合**

这是 Transformer 结构灵活性的体现。它不关心向量在序列中的“绝对位置”关系，只关心向量之间的“内容关系”。这也意味着，如果你需要模型感知位置（比如处理自然语言时，语序很重要），就必须在输入时额外加上 **位置嵌入（Positional Encoding）**，因为自注意力机制本身是置换等变的。

这张图中展示的 Transformer 模块，是现代大模型如 BERT、GPT 等的最基本单元，不同的模型只是在这个 Block 的基础上做了微调，比如改变层数、改变注意力机制或增加掩码。

#### vision Transformers(ViT)

Vision Transformer 将 Transformer 架构直接应用到了图像分类任务上。它的核心思想非常直接：**把一张图像当作一个序列来处理**。既然 Transformer 在处理自然语言序列上表现极其出色，那么只要把图像转化成序列，它同样能发挥作用。

![ViT](../img/ViT.png)

**图像分块与线性投影**

标准 Transformer 的输入是一组向量，为了处理图像，ViT 首先将输入图像切分成多个固定大小的图像块（Patches），每个图像块的形状是 $3 \times 16 \times 16$，对应 RGB 三通道和 16x16 的空间尺寸。

这些图像块被展平后，经过一个**线性投影层**，映射成 $D$ 维的向量。这一步本质上相当于用一个卷积层（卷积核大小和步长均等于 16）来提取特征，将每个图像块转换成序列中的一个“词向量”。

**位置嵌入（Positional Embedding）**

图像被切分成序列后，**丢失了原有的二维空间结构信息**。Transformer 不知道哪个图像块在左上角，哪个在右下角。因此，必须显式地将位置信息注入到输入中。ViT 引入了**位置嵌入**，这是一个可学习的 $D$ 维向量，每个位置对应一个。将其与图像块的线性投影向量相加，模型就能在后续的注意力计算中感知到各图像块的相对位置。

**分类令牌（Class Token）**

在输入序列的最前面，ViT 额外添加了一个可学习的向量，称为**分类令牌**。它本身不包含图像信息，但作为序列的一部分参与 Transformer 的全局注意力计算。经过多层 Transformer 处理后，它能够聚合整个图像所有块的信息，最终成为图像的整体表示。

**Transformer 编码器**

这组向量（包括分类令牌和所有图像块向量）被送入 Transformer 模块。这部分的计算与处理文本的 NLP Transformer 完全一致，包括多头自注意力、MLP、残差连接和 LayerNorm，没有任何针对图像的特殊改动。这展现了 Transformer 极强的架构通用性。

**分类输出**

经过 Transformer 处理后，序列中各个位置都会输出对应的向量。如何将这些输出向量汇总成最终的图像表示，通常有两种主流方案。

- 第一种方案是引入一个特殊的**分类令牌**，也就是提到的 “Special extra input: classification token (D dims, learned)”。它在输入序列的最前面加入，本身不包含图像信息，但作为序列的一部分参与 Transformer 的全局注意力计算。经过多层 Transformer 处理后，它能够聚合整个图像所有块的信息。

    最终，只取出这个分类令牌对应的输出向量，送入一个简单的线性层，进行 “Linear projection to C-dim vector of predicted class scores”，也就是映射到 $C$ 维（$C$ 为分类类别数），再经过 Softmax 得到分类概率。这种做法最初借鉴自 NLP 领域的 BERT 模型，优点在于模型可以通过注意力机制，专门训练一个用于汇总全局信息的向量。

- 第二种方案是不引入额外的分类令牌，而是对 Transformer 输出的所有图像块向量进行池化，通常是全局平均池化（Global Average Pooling）。池化操作会在序列长度这一维度上对所有向量求平均，得到一个全局特征向量。然后，再进行一次线性投影，将这个全局特征映射到 $C$ 维的类别空间，最后通过 Softmax 得到分类概率。这种方案直接用所有图像块的特征来汇总信息，没有额外的可学习参数，结构更加简单直接。


两种方案在实践中都能取得很好的效果，选择哪一种往往取决于具体任务的复杂度和模型设计偏好。


##### 优化Transformer


原始的 Transformer 架构自 2017 年以来变化不大，但为了提升训练稳定性与模型表达能力，现代大模型普遍采用了几项关键的架构优化。

**Pre-Normalize（前归一化）**

在原始的 Transformer 中，层归一化（LayerNorm）被放置在残差连接**之后**，即 `Output = LayerNorm(x + SubLayer(x))`，这被称为 Post-Norm。Post-Norm 在深层网络中训练不稳定，容易导致梯度消失，需要精细的学习率预热策略。

现代 Transformer 普遍采用 Pre-Norm，将层归一化移到残差连接**之前**，即 `Output = x + SubLayer(LayerNorm(x))`。其有效性在于，残差路径上没有归一化操作的干扰，梯度可以更直接地流过恒等路径，这使得模型在初始阶段梯度表现良好，无需学习率预热即可稳定训练，且收敛速度更快。

**RMSNorm（均方根归一化）**

RMSNorm 是 Pre-Norm 的轻量化改进版，移除了 LayerNorm 中的**均值中心化**步骤，仅保留**方差归一化**（通过均方根 RMS 进行缩放）。其核心公式为 `RMSNorm(x) = γ * x / RMS(x)`，其中 `RMS(x) = sqrt(1/d * Σ x_i^2)`。

这种简化减少了约 50% 的计算量，并能带来 7% 到 64% 的速度提升，同时保持与 LayerNorm 相当的下游性能。此外，RMSNorm 还能自然地惩罚过大的激活值，具有隐式的学习率自适应能力。GPT-3、LLaMA、Mistral 等模型均采用 RMSNorm。

**SwiGLU MLP（门控非线性 MLP）**

标准 MLP 的参数量为 `2 * d * h`（d 为输入维度，h 为隐藏层维度）。SwiGLU 引入了**门控机制**，使用两条并行分支：一条提供候选特征（线性路径），另一条决定通过多少（门控路径），两者逐元素相乘后再降维。

其公式为 `SwiGLU(x) = (Swish(xW_gate) ⊙ xW_up) W_down`。由于升维阶段需要两个投影矩阵（`W_gate` 和 `W_up`），参数量变为 `3 * d * h`。为了保持与标准 MLP 参数量相当，通常将隐藏层维度 `h` 调整为 `8d/3` 而非 `4d`。这种门控机制增强了模型的非线性表达能力，被 LLaMA 等模型广泛采用。

**Mixture of Experts（MoE）**

MoE 将 Transformer 中的 MLP 层替换为**多组独立的 MLP**，每组称为一个“专家”。每个 token 通过一个可学习的**路由网络**，被分配到少数几个（A 个）专家进行计算，而不是全部专家。

这种方式的核心优势在于：**参数量随专家数量 E 成倍增加，但计算量仅随激活专家数 A 线性增加**。例如，设置 E=8，每个 token 只激活 A=2 个专家，模型总参数量增加 8 倍，但每个 token 的计算量只增加 2 倍。这使得模型可以拥有超过万亿的参数，而推理成本却相对可控。GPT-4o、Claude 3.7、Gemini 2.5 Pro 等当今最大的 LLM 几乎都使用了 MoE 架构。


#### 语义分割 Semantic Segmentation

语义分割的目标是**对图像中的每一个像素进行分类**，输出一张与输入尺寸相同的标签图。与图像分类输出单一标签不同，它需要输出像素级别的密集预测。

**全卷积网络（FCN）** 是语义分割的基础架构。它去掉了传统分类网络末端的全连接层，全部使用卷积层，从而使网络可以接受任意尺寸的输入，并输出空间维度的特征图。在特征提取过程中，网络会通过下采样不断缩小空间尺寸并增加通道数，从而扩大感受野，提取更高层次的语义信息。

**下采样**指的是通过步幅卷积或池化操作，降低特征图的空间分辨率。

**上采样**则是将低分辨率的特征图恢复到原始图像的尺寸。常见的上采样方式有几种：
- **反池化操作**：最近邻法直接复制像素值，床钉法则在固定位置填入数值。
- **最大反池化（Max Unpooling）**：在池化时记录下最大值所在的位置坐标，在上采样时将数值填回这些坐标，其他位置补零。
- **转置卷积（Transposed Convolution）**：也被称为可学习的上采样，让网络自己学习如何放大特征。

**U-Net** 引入了编码器-解码器结构与**跳跃连接（Skip Connections）**。**跳跃连接**是一种将编码器阶段的高分辨率特征图直接复制并拼接到解码器对应层的操作。由于下采样会丢失部分空间细节，这种跳跃连接使解码器在恢复空间信息时，能够同时利用低层的纹理细节和高层的语义信息。

#### 目标检测 Object Detection

目标检测不仅需要识别图像中物体的类别，还需要确定它们的位置，输出包含类别标签和边界框坐标的集合。

早期的做法是将分类和定位结合，使用**多任务损失（Multitask Loss）** 来同时优化分类误差和边界框回归误差。但这种方法通常只适用于单一物体，对于包含多个物体的图像，扩展性很差。

为了解决这个问题，研究者提出了**区域提议（Region Proposals）** 的方法，并经历了一系列演进：
- **R-CNN**：通过选择性搜索生成大量候选区域，然后逐一送入网络进行分类和边界框修正。
- **Fast R-CNN**：通过共享卷积计算，避免了重复提取特征。
- **Faster R-CNN**：进一步引入了**区域提议网络（RPN）**，让网络自己学习生成候选区域，实现了端到端的训练。

**后处理**
在检测过程中，通常还需要对预测结果进行后处理：
- **阈值处理**：保留概率最高的前 K 个候选区。
- **非极大值抑制（NMS）**：一种后处理算法，它会抑制掉与最高分框重叠度过大的其他框，最终只保留最优结果。

##### Faster R-CNN 

Faster R-CNN 流程如下：
1. **特征提取**：输入图像首先经过 CNN 骨干网络，提取出**特征图（feature map）**。这个特征图同时被后面两个分支共享。
2. **区域提议网络（RPN）**：特征图首先输入 RPN。RPN 在特征图的每个位置上生成多个不同尺度和比例的**锚框（Anchors）**，并输出两样东西：一是分类分数，判断该锚框是前景还是背景；二是边界框回归偏移量。随后，通过**分类损失（Classification loss）**和**边界框回归损失（Bounding-box regression loss）**对 RPN 进行训练。最终，RPN 输出一组高质量的**候选区域（proposals）**。
3. **RoI Pooling**：将 RPN 生成的候选区域和 CNN 特征图结合，进行**感兴趣区域池化（RoI pooling）**。由于候选区域大小不一，RoI pooling 会将其统一池化为固定尺寸的特征图，以便送入后续的全连接层。
4. **检测头与最终预测**：池化后的特征图经过全连接层，最终输出两个结果：一是每个候选区域的类别分类（Classification loss），二是对该候选区域边界框的进一步精确回归（Bounding-box regression loss）。

**训练机制与层次**

Faster R-CNN 的核心在于如何训练 RPN 和检测头。由于两者共享前面的卷积层，直接一起训练容易导致不稳定。因此，早期的经典实现采用了**4 步交替训练（Alternating Training）**：

1. **训练 RPN**：使用预训练模型初始化共享卷积层，单独训练 RPN，使其学会生成高质量的候选区域。
2. **训练检测头**：利用第 1 步训练好的 RPN 生成的候选区域，作为输入来训练 Fast R-CNN 的检测头。此时共享卷积层也会被微调。
3. **微调 RPN**：固定共享卷积层，只使用第 2 步中更新过的特征，微调 RPN 独有的层，让 RPN 适应新的共享卷积特征。
4. **微调检测头**：再次固定共享卷积层，使用第 3 步微调后的 RPN 生成候选区域，仅微调检测头独有的层。

经过这 4 步，RPN 和检测头都能在共享特征的基础上达到良好的协作效果。

---
##### 单阶段检测器：YOLO (You Only Look Once)

YOLO 彻底抛弃了区域提议（Region Proposal）的路线，将目标检测重新定义为一个**单一的回归问题**。它直接从图像像素映射到边界框坐标和类别概率，实现了极致的速度和端到端的训练。

**核心机制：网格划分与张量输出**

YOLO 将输入图像划分为 $S \times S$ 的网格（Grid）。如果某个真实物体的中心落在某个网格内，该网格就负责检测这个物体。

最终网络的输出是一个形状为 $S \times S \times (B \times 5 + C)$ 的张量，其含义拆解如下：
- $S \times S$：特征图的网格数量。例如 YOLOv1 使用 $7 \times 7$ 的网格。
- $B$：每个网格预测的边界框（Bounding Box）数量。YOLOv1 中 $B=2$。
- $5$：每个边界框包含 5 个预测值，分别是 $x, y, w, h$ 和**置信度（Confidence）**。
- $C$：每个网格预测的条件类别概率。

**坐标与置信度**

- **坐标 $x, y, w, h$**：$x, y$ 是边界框中心相对于该网格左上角的**偏移量**（归一化到 0-1 之间）。$w, h$ 是边界框相对于**整张图像**宽高的比例。这使得预测坐标具有尺度不变性。
- **置信度（Confidence）**：公式为 $\Pr(\text{Object}) \times \text{IoU}_{\text{pred}}^{\text{truth}}$。它包含两层信息：
    1. $\Pr(\text{Object})$：该网格是否包含物体的概率。
    2. $\text{IoU}$：预测框与真实框的交并比。如果没有物体，置信度理论上应该为 0。
- **条件类别概率**：每个网格预测 $C$ 个类别概率，表示**在包含物体的前提下**，该物体属于某个类别的概率（与置信度独立计算）。

**损失函数**

YOLO 的损失函数由多个部分加权相加，这也是它训练的难点与精髓所在：
1. **坐标损失**：对边界框的 $x, y, w, h$ 进行回归（通常使用均方误差 MSE）。注意，对 $w, h$ 开根号可以弱化大物体和小物体对损失的权重差异。
2. **置信度损失**：包含物体的框，置信度要靠近预测框与真实框的 IoU；不包含物体的框，置信度要接近 0。因为一张图大部分网格没有物体，所以需要引入一个超参数 $\lambda_{\text{noobj}}$（通常取 0.5）来降低无物体框的损失权重，防止模型偏向预测背景。
3. **分类损失**：负责预测物体的网格，需要对其类别概率进行监督。
4. **责任分配机制**：若一个网格预测了多个框（$B>2$），在训练时，只选择与真实框 IoU 最大的那个预测框来负责该物体，另一个框不计算坐标损失，只计算置信度损失。

**后处理**

在测试时，YOLO 一次前向传播直接输出所有预测。后处理包含两个关键步骤：
1. **阈值处理**：将每个框的置信度与类别概率相乘，得到特定类别的置信度分数（$\text{Class-Specific Confidence}$）。过滤掉低于阈值（如 0.5）的预测。
2. **非极大值抑制（NMS）**：由于同一个物体会在相邻网格产生多个预测，需要使用 NMS 去除冗余。它保留得分最高的框，并抑制掉与它重叠度（IoU）超过设定阈值的其他框。


- **优点**：一步到位，无需生成候选区域，速度极快（YOLOv1 在 GPU 上可达 45 FPS），非常适合实时任务；由于在整张图上训练，背景误检率较低。
- **缺点**：
    - **空间限制**：每个网格只能预测一个物体。当多个物体的中心落在同一网格时（如一群鸟、一堆重叠的人），模型无法区分。
    - **小物体不友好**：网格较粗糙（$7 \times 7$），小物体的中心容易落到背景网格中，导致漏检。
    - **定位精度**：直接回归坐标没有两阶段方法的“二次微调”，初期定位精度不如 Faster R-CNN。



#### 基于 Transformer 的目标检测 DETR

![DETR](../img/DETR.png)

DETR 将目标检测视为一个**集合预测问题**，直接使用 Transformer 架构来完成检测，彻底抛弃了传统检测中所需的锚框和非极大值抑制，实现了真正的端到端。它的核心机制建立在全监督学习的基础上，训练过程需要图像对应的真实标签（类别和边界框）。

**输入**

模型的输入是一张图像，处理流程如下：
1. 图像首先经过 CNN 骨干网络（如 ResNet）提取特征，得到形状为 $(B, C, H', W')$ 的特征图，其中 $B$ 是批次大小，$C$ 是通道数。
2. 特征图被展平为 $(B, H'W', C)$，即序列长度为 $H'W'$ 的特征向量集合。
3. 加上位置编码后，送入 Transformer 编码器，通过全局自注意力捕捉图像上下文信息。
4. 解码器的输入除了编码器输出外，还包含一组可学习的**物体查询（Object Queries）**。假设模型预设输出 $N$ 个预测结果（通常 $N=100$），那么就会初始化 $N$ 个查询向量，形状为 $(B, N, C)$。这些查询在解码器中通过交叉注意力机制，主动去“询问”图像中哪里存在物体。

**输出**

解码器输出的 $N$ 个查询向量，每个都会独立送入一个共享权重的**前馈网络（FFN）**，也就是预测头。预测头分为两条分支：
- **分类分支**：输出 $C+1$ 个类别的分数，包含 $C$ 个真实类别和 1 个“无对象”（`no object`）类别。
- **回归分支**：输出归一化的边界框坐标 $(x_{center}, y_{center}, w, h)$，共 4 个值。

最终模型的输出是两个张量：分类分数 $(B, N, C+1)$ 和边界框坐标 $(B, N, 4)$。

**全监督与二分匹配损失**

DETR 采用**全监督训练**，核心在于其独特的匹配和损失计算流程。

1. **匈牙利匹配（Hungarian Matching）**。模型输出的 $N$ 个预测结果，会与图像中 $M$ 个真实标签进行一对一匹配。这是一个二分图匹配问题，通过匈牙利算法寻找总匹配代价最小的方案。匹配代价由分类概率、边界框回归误差（L1）和 GIoU 误差共同决定。

2. **计算二分匹配损失（Bipartite Matching Loss）**。匹配成功的预测负责计算类别损失和边界框损失（通常为 L1 Loss 和 GIoU Loss 的组合）。未匹配的预测被强制分配为 `no object`，只计算分类损失中的背景类损失，不计算边界框回归损失。

这种一对一匹配机制确保了每个真实物体只对应一个预测，从而**无需 NMS 后处理**，直接输出唯一的检测结果。

**局限性**
DETR 的训练非常依赖长周期。因为从零开始学习“哪里的查询对应哪个物体”非常困难，通常需要 500 个 Epoch 才能收敛，远长于传统检测器。这也导致了 DETR 对**小物体**的检测性能欠佳，因为 Transformer 的全局注意力在初期很难精确定位微小目标。

#### 实例分割 Mask R-CNN

**Mask R-CNN** 是在 Faster R-CNN 基础上扩展而来的实例分割模型。**实例分割（Instance Segmentation）** 不仅需要像目标检测那样输出边界框和类别，还需要对每个物体生成像素级别的掩码，区分出属于该物体的具体像素区域，即便是同类物体也能区分开。

它的核心改进在于：
- 在原有的 Faster R-CNN 检测头（分类 + 边界框回归）之外，**额外增加了一个掩码分支**。这个分支通常是一个小型全卷积网络（FCN），接收 RoI 特征后，输出一个 $K \times m \times m$ 的掩码（$K$ 为类别数，$m$ 为掩码分辨率）。
- 引入了 **RoIAlign** 替代原来的 RoI Pooling。RoI Pooling 在量化坐标时存在误差，会对像素级的掩码预测造成负面影响。RoIAlign 使用双线性插值，避免了量化误差，使空间对齐更精确。

训练时，Mask R-CNN 同样使用多任务损失，将分类损失、边界框回归损失和掩码分割损失相加。预测时，先通过检测头找到物体，再由掩码分支生成对应物体的掩码。


---



#### 模型可视化与可解释性

深度学习模型通常是一个黑盒，为了理解模型做出分类决策时到底“看”了图像的哪些区域，研究者开发了多种可视化技术。早期的方法主要用于卷积神经网络（CNN），而现代 Transformer 架构则天然自带激活图。

**类激活映射 CAM**

**类激活映射（Class Activation Mapping, CAM）** 是一种可视化技术，用于理解 CNN 在分类时关注的区域。它的实现依赖特定的网络结构，通常要求网络末端使用**全局平均池化（Global Average Pooling, GAP）** 代替全连接层，再接一个全连接分类层。**全局平均池化**会将最后一个卷积层输出的每个通道特征图求平均，得到一个向量，再通过分类层得到每个类别的分数。

计算 CAM 时，取出最后一个卷积层的 $C$ 张特征图，以及分类层中对应某个类别的权重。将每张特征图与对应的权重相乘并求和，就得到了该类别在输入图像上的热力图。热力图中的高亮区域，就是模型认为与该类别最相关的区域。但 CAM 的局限性很明显：它要求网络架构必须是 GAP + 全连接层的特定结构，对于没有这种结构的预训练模型（如 VGG、ResNet 原始版）无法直接应用。

**梯度加权类激活映射 Grad CAM**

**Grad CAM（Gradient-weighted Class Activation Mapping）** 是 CAM 的推广版本，它不需要修改网络结构，适用于任意卷积神经网络。

与 CAM 直接使用分类层权重不同，Grad CAM 利用**梯度**来作为特征图的权重。具体流程如下：
1. 前向传播图像，得到特定类别的分类分数。
2. 反向传播，计算该类别分数相对于最后一个卷积层特征图的梯度。
3. 对每个特征图的梯度进行全局平均池化，得到每个通道的权重。
4. 用这些权重对特征图进行加权求和，再经过 ReLU 激活，最终得到热力图。

使用 ReLU 的原因是，我们只关心对目标类别有正向贡献的区域，负值在可视化时会被丢弃。Grad CAM 生成的定位图虽然分辨率较低，但它能直观展示模型在判断某个类别时，注意力集中在图像的哪个部分，是 CNN 时代非常实用的可解释性工具。

**Transformer 自带的注意力图**

在卷积网络中，获取模型关注区域需要依赖 CAM 或 Grad CAM 等后处理手段。但在 Transformer 中，特别是用于视觉任务的 Vision Transformer，**注意力图是模型架构的天然副产品**，这为模型的可解释性提供了极大的便利。

在 ViT 中，输入图像被切分为多个图像块（Patches），每个图像块经过线性投影后，转化为一个向量，作为序列的一部分进入 Transformer 编码器。

在自注意力层中，每一个查询向量（Query）都会与所有的键向量（Key）计算相似度，再经过 Softmax 得到注意力权重矩阵。这个注意力权重矩阵本身就是一个天然的**激活图**。

矩阵的每一行代表一个特定的图像块（查询），每一列代表它对该图像中其他所有图像块的关注程度。将某个图像块对应的注意力权重按空间位置排列回二维图像，就能直观地看到该图像块在关注图像的哪些区域。

与 CAM / Grad CAM 的区别

- **CAM / Grad CAM**：面向 CNN 的后期分析工具，或要求特定的网络结构（GAP + 全连接），或依赖反向传播的梯度加权，属于对模型的“外部干预”。
- **Transformer 注意力图**：前向传播时自然产生的中间结果，无需额外的网络修改，也无需计算梯度，直接将注意力矩阵可视化即可。

在实际应用中，可视化 ViT 的注意力时通常需要做额外处理。因为 ViT 有多头注意力（Multi-head Attention），每个头关注的特征不同。

有的头可能关注局部的纹理细节，有的头可能关注全局的语义关联，甚至有的头会关注背景。因此，通常会**对所有注意力头求平均**，或者选取特定的头进行观察。

此外，关于“注意力是否等同于解释”在学术界仍有讨论。注意力权重高，意味着模型在融合信息时参考了这个位置，但不一定代表这个位置是模型做出分类决策的“唯一原因”。不过，从实用的可视化角度看，注意力图确实为理解 Transformer 的决策过程提供了最直接、最轻量的观察窗口。

---


#### 视频理解 Video Understanding

视频本质上是**2D图像加上时间维度**，也就是连续的图像帧序列。视频分类任务就是给一个视频片段，预测其所属的类别。视频理解的难点在于，不仅要识别每一帧里的**空间信息**（外观、场景、物体），还要捕捉帧与帧之间的**时间信息**（运动、动作变化）。


**Training on Clips**

![ToC](../img/TrainOnClips.png)

原始的原始视频通常非常长，且帧率很高，直接送入网络计算量巨大且内存无法承受。因此，标准的训练流程是将视频裁剪成片段来处理。

- **训练阶段**：从长视频中抽取**低帧率**的短片段，让模型对这些短片段进行分类。
- **测试阶段**：在长视频的不同时间段提取多个重叠或独立的片段，分别送入模型预测，然后**对所有片段的预测结果取平均**，作为整个视频的最终预测。这有效解决了长视频预测的问题。

**视频特征提取的早期策略**

视频体积很大，为了减少计算量，早期常用**压缩时间空间**、**抽帧**、**降低分辨率**等方法。在将视频送入网络前，主要有几种融合策略：

- **简单帧 CNN（Simple Frame CNN）**：如果画面变化不大，可以跑单帧，将每一帧的预测结果平均。
- **后融合（Late Fusion）**：让2D CNN分别处理每一帧，把最后得到的特征图展开成大特征向量，再在全连接层进行融合。这种方法的缺点是**全连接层参数太大**。改进一步的方法是，对 $T$ 帧的特征直接做池化，再映射到输出，但池化操作也可能丢掉一些重要的时间特征。
- **早期融合（Early Fusion）**：将时间维度的帧拼接成一个很长的输入，然后让一个2D CNN处理。直觉上这能包含低层级的运动信息，但2D CNN很难处理这种维度，且由于时间维度被完全展平，**失去了时间不变性**（对不同时间位置需要不同的过滤器）。

#### 3D CNN

针对早期融合（Early Fusion）的缺陷，**3D CNN** 是视频理解的重要里程碑，它能够**自然且端到端地处理时空信息**。

**机制细节**：
- **输入**：四维张量 $(C, T, H, W)$，分别代表通道数、时间长度、高度、宽度。
- **卷积核**：2D CNN 的卷积核是 $k_h \times k_w$，只能在空间滑动；而 3D CNN 的卷积核是 $k_t \times k_h \times k_w$，它在空间和时间维度上同时滑动。
- **维度变化**：经过3D卷积和池化，时间维度 $T$ 会和空间维度 $H, W$ 一样被逐步压缩。浅层的3D卷积感受野较小，捕捉的是短时间内的细微动作；深层的卷积感受野在时空上同时扩大，能捕捉较长时间段的复杂动作。

**代表工作**：VGG of 3D CNNs。在 Sports-1M 等大规模视频数据集上训练，证明了3D CNN在捕捉时空特征上的有效性。但缺点是计算量是2D CNN的数倍，参数量急剧增加。


##### 双流网络 Two-Stream Networks

![TS](../img/Two_Stream.png)

视频包含外观和运动两部分信息，双流网络将它们分开处理，再融合结果。

- **空间流（Spatial Stream）**：
  输入是**单帧RGB图像**。使用标准的2D CNN，负责识别物体的外观、场景和静止的物体。
- **时间流（Temporal Stream）**：
  输入是**堆叠的光流图像**。**光流（Optical Flow）** 是一种用来测量像素在相邻帧之间运动变化的特征（通常包含 $x$ 方向和 $y$ 方向两个通道）。它直接刻画了运动信息，剥离了外观信息。堆叠多帧的光流图（如 $[2 \times (T-1) \times H \times W]$）送入2D CNN，提取运动特征。
- **融合**：两个网络独立输出分类分数，通过**类别分数融合（Class Score Fusion）** 得到最终结果。常用的融合方式是直接平均，或者在验证集上训练一个权重进行加权平均。

##### I3D 架构：迁移架构（Inflated 3D ConvNets）

双流网络存在两个独立的网络，且需要预计算光流（耗时）。**I3D 架构** 的核心思想是“**迁移架构**”。

- **为什么需要迁移**：视频数据集通常比 ImageNet 小得多，从零训练3D CNN 极易过拟合。
- **如何实现“膨胀（Inflation）”**：将成熟的 2D 卷积核（如 $3 \times 3$）“膨胀”为 3D 卷积核（如 $3 \times 3 \times 3$）。
- **关键技巧**：为了保持激活值的方差不变，膨胀后的 3D 卷积核权重需要除以时间维度 $k_t$（比如把 $3 \times 3$ 的权重除以 3，复制到 $3 \times 3 \times 3$ 的时间维度上）。这样，就可以直接用在 ImageNet 上预训练好的 2D 权重来初始化 3D 模型，极大加速了训练。


**时空自注意力：非局部块（Nonlocal Block）**

![NB](../img/NonlocalBlock.png)

标准 3D CNN 虽然能捕捉时空信息，但感受野是局部且有限的。为了建模长距离的时空依赖，引入了**非局部块**。


1. **输入特征**：$(C \times T \times H \times W)$。
2. **生成QKV**：输入分别经过 $1\times1\times1$ 卷积，得到 Queries、Keys、Values，维度均为 $(C' \times T \times H \times W)$。
3. **计算注意力权重**：将 Queries 转置后与 Keys 进行矩阵乘法，得到维度为 $(THW \times THW)$ 的注意力权重矩阵，然后进行 Softmax 归一化。
4. **加权求和**：用注意力权重对 Values 进行加权求和，得到 $(C' \times T \times H \times W)$ 的输出。
5. **残差连接**：最后经过 $1\times1\times1$ 卷积映射回原通道数，与原始输入进行相加（残差连接）。

这本质上就是**时空自注意力**。它让视频中的任意一个时空位置（像素点），都能直接与视频中所有其他位置交互，完美解决了3D CNN局部感受野的限制。

**其他演进方向**

- **循环卷积神经网络（ConvLSTM）**：用二维卷积替代了循环网络中的矩阵乘法，以便在处理特征图时保留空间结构。但 RNN 的序列依赖导致它很难并行化，所以在大规模训练中用的并不多。
- **时空检测任务**：视频理解不仅仅局限于分类，还包括在视频中检测动作发生的时间和空间位置（如动作定位）。
- **前沿应用**：如视觉引导的声音分离，利用视觉信息辅助分离视频中的混合声音，这是多模态学习的重要方向。

---

#### 大规模分布式训练
Llama3-405B

GPU硬件
H100的三级存储层级


128 FP32 Cores

4 Tensor Cores

多GPU训练
一是计算。
拆分计算
二是通信

并行方式（主要是五种）
DP，CP，PP，TP

#### 自监督学习 Self-Supervised Learning

训练深度神经网络通常依赖大量的人工标注数据，比如语义分割需要逐像素的标注，成本极高且难以规模化。自监督学习（SSL）的核心思路是：**利用大量无标签数据，通过构造预训练任务来自动生成监督信号，让模型学到通用的特征表示，然后再用少量带标签的数据迁移到下游任务上。**

**核心框架与术语**

自监督学习包含两个阶段：
- **预训练阶段（Pretext Objective）**：输入大量无标签数据，通过一个设计好的预训练任务，训练一个编码器来提取特征。预训练任务通常需要编码器学习到图像的结构、语义或上下文信息。
- **下游阶段（Downstream Objective）**：将预训练好的编码器迁移到有标签的小规模数据集上，进行微调或冻结权重做线性探测。

**如何评估自监督学习的效果？**

评估自监督学习是一个多维度的任务。由于我们真正关心的是“学到的特征好不好”，而不是预训练任务本身的表现，因此评估通常围绕以下几个指标展开：

1. **线性探测（Linear Probing）**
   这是最常用的评估协议。具体操作是冻结预训练好的编码器，仅在其后接入一个浅层线性分类器，使用少量有标签的目标数据训练这个分类器。如果编码器学到的特征具有高度的线性可分性，线性分类器就能取得优异表现。其数学表达为：冻结编码器 $f_\theta(x)$（$\theta$ 不更新），仅优化线性分类器 $W$：
   $$ \hat{y} = \text{softmax}(W f_\theta(x)) $$
   $$ \mathcal{L}(W) = -\frac{1}{N} \sum_{i=1}^N \sum_{c=1}^C y_{i,c} \log(\hat{y}_{i,c}) $$

2. **微调（Fine-tuning）**
   与线性探测不同，微调不冻结编码器，而是用少量标注数据对整个网络进行端到端的微调。这能反映编码器在特定任务上的最高性能上限，但也容易受到过拟合的影响。

3. **特征聚类与可视化**
   - **聚类指标**：直接对编码器输出的特征进行聚类（如 K-Means），用聚类准确率、归一化互信息（NMI）等指标衡量特征的内在结构。如果同类物体在特征空间中形成紧密的簇，说明特征质量很高。
   - **t-SNE / UMAP 可视化**：将高维特征降维到二维空间进行可视化，直观判断不同类别的特征是否分离。

4. **鲁棒性与泛化能力**
   评估编码器在输入受到干扰（如旋转、遮挡、噪声）时，其特征表示是否依然稳定。通常会在不同数据集或不同数据分布下进行迁移测试，看性能下降的幅度。

5. **计算效率与预训练代价**
   自监督学习往往需要极长的训练周期。评估时也需要关注预训练所需的计算资源（GPU 小时数）以及模型推理时的速度，这决定了该方法是否具备实际落地的可行性。

6. **下游任务的最终表现**
   无论中间指标多好，最具有说服力的是在目标检测、语义分割等下游任务上的最终精度。这是衡量特征迁移能力的金标准。



**常见的预训练任务（Pretext Tasks）**

为了自动生成标签，可以通过对图像进行已知的变换来构造任务。
- **旋转预测（Rotation Prediction）**：将图像随机旋转 $0°, 90°, 180°, 270°$，让网络预测旋转角度。这个任务背后的假设是：**只有当模型具有对物体的视觉常识时，它才能意识到图片被旋转了多少角度。** 网络若想判断图像是否倒置，就必须学习到物体的结构和重力方向等高层语义特征。这个任务在 CIFAR-10 上进行过大量验证，并证明能学到丰富的注意力区域。
- **拼图任务（Jigsaw Puzzles）**：将图像切分为网格 patch （比如3*3的网格）并随机打乱，让网络预测正确的排列顺序。这迫使模型理解局部 patch 与整体结构的空间关系。
- **图像补全（Inpainting）与掩码（Masking）**：这是目前最常用的预训练任务之一。随机遮挡图像的一部分，让网络根据上下文补全缺失内容。**Masked Auto Encoders（MAE）** 就是基于重建的框架，其损失函数通常为均方误差（MSE），直接优化重建像素与真实像素的差异：$\mathcal{L} = \mathbb{E}_{x}[\| \hat{x} - x \|_2^2]$。它迫使编码器理解图像的内在结构和语义。

- **图像着色（Colorization）：基于颜色的自监督方法利用了图像的通道信息。核心思想是**跨通道预测（cross-channel predictions）**。将图像转到 LAB 颜色空间，已知亮度通道 L，让网络预测颜色通道 a 和 b。预测颜色要求模型理解物体语义，比如草通常偏绿、天空通常偏蓝。



**Simple Exam**

![HowToEvaluate](../img/SSL.png)


评估自监督学习的最简单流程包含两个明确的阶段：

1. **自监督预训练阶段**：输入海量无标签数据（lots of unlabeled data），通过自监督学习训练一个特征提取器（如 CNN）。
2. **下游迁移与评估阶段**：丢弃预训练任务专用的输出层，保留特征提取器。然后接入一个浅层网络，使用少量有标签的目标数据来训练这个浅层网络，最后在目标任务上评估效果。


以“旋转预测”（predicting image rotations）为例，自监督训练的具体操作步骤如下：

**1.自动构造标签**

从无标签数据集中取出一张图片（如图中的鸟）。对这张图片施加四种已知的旋转操作之一：0°、90°、180°、270°。旋转的角度本身，就成为了这张图片的标签。这个过程完全不需要人工参与，标签是凭空自动生成的。这就是“自监督”的本质——监督信号来源于数据本身。这里本质为图像分类任务。

**2.搭建网络并前向传播**

网络结构如图所示，由卷积层（`conv`）和全连接层（`fc`）组成。卷积层作为特征提取器，将输入图片转化为特征向量；全连接层作为任务头，将特征映射到四个类别的预测分数。

假设输入图片为 $x$，真实旋转角度类别为 $y \in \{0,1,2,3\}$。前向传播过程为：
$$
\hat{y} = \text{softmax}(W_{\text{fc}} \cdot f_{\text{conv}}(x))
$$
其中 $f_{\text{conv}}(x)$ 是卷积层提取的特征，$W_{\text{fc}}$ 是全连接层的权重，$\hat{y}$ 是预测出的四个旋转类别的概率分布。

**3.计算损失并反向传播**

使用标准的交叉熵损失来衡量预测与真实旋转标签的差异：
$$
\mathcal{L} = -\sum_{c=0}^{3} y_c \log(\hat{y}_c)
$$
然后通过反向传播，同时更新卷积层和全连接层的所有参数。这里的关键在于，网络为了能在四选一的任务中答对，必须去理解图片的内容——比如鸟的头部和爪子的相对位置。如果图片被倒置，正常的语义结构就被破坏了。因此，这个看似简单的任务，迫使网络学习到关于物体结构、重力方向、语义部件的高层视觉常识。

**4.丢弃任务头，保留特征提取器**

预训练完成后，全连接层（`fc`）的使命就结束了，丢弃它。我们真正要保留的，是那个通过旋转任务训练出来的卷积层（`conv`）。这个卷积层就是“特征提取器”，它包含了模型从无标签数据中学到的全部通用视觉知识。

**下游迁移与线性探测**

在评估阶段，将预训练好的卷积层冻结（参数不再更新），在后面接一个全新的线性分类器。输入带标签的目标数据（比如一张鸟的图片，标签为“鸟”），只训练这个新的线性分类器，损失函数同样为交叉熵：
$$
\mathcal{L}(W) = -\sum_{c=1}^{C} y_c \log(\text{softmax}(W f_\theta(x))_c)
$$
其中 $f_\theta(x)$ 是冻结的预训练卷积特征，$W$ 是线性分类器的权重。

**为什么用浅层网络？**

如果特征足够好，它们在特征空间中应该是**线性可分**的。此时，简单的浅层线性分类器就能轻易画出分类边界。相反，如果特征质量差，就必须使用复杂的分类器才能拟合数据，这就掩盖了特征提取器的缺陷。因此，浅层网络上的高准确率，是证明编码器学到了优质表征的最有力证据。



#### 基于重建与生成的预训练任务

**拼图任务（Jigsaw Puzzles）**

拼图任务的要点是：将图像切分为网格 patch，随机打乱顺序，让网络预测正确的排列。其原理在于，为了正确还原顺序，网络必须理解局部 patch 与整体结构的空间关系。这个任务的难点在于输出空间极其巨大（如 9 个 patch 的全排列是 $9!$），实际训练中通常需要限制候选排列（如只考虑几百种相对位置组合）来降低难度。

**图像着色（Colorization）**

图像着色的核心操作是：**将图像的颜色空间转换到 LAB 空间，将亮度通道 L 作为输入，让网络预测颜色通道 a 和 b。** 与 RGB 不同，LAB 空间把亮度（L）与颜色（a 和 b）解耦了。

其操作步骤如下：
1. 将 RGB 图像转换到 LAB 色彩空间。
2. 分离出 L 通道作为网络输入（因为人眼对亮度更敏感，且 L 通道包含了完整的结构信息）。
3. 网络输出对 a 和 b 两个颜色通道的预测。
4. 计算预测的 ab 与真实的 ab 之间的 L2 损失（均方误差）：
   $$ \mathcal{L} = \| \hat{ab} - ab \|_2^2 $$
5. 反向传播，更新网络参数。

**原理与多模态问题**：
预测颜色的任务迫使网络理解物体语义。例如，草通常偏绿，天空通常偏蓝，老虎有橙色和黑色的条纹。网络为了正确上色，必须从 L 通道中提取出形状、边缘、纹理和物体类别等高层特征。
然而，这个方法存在一个隐患：**多模态不确定性**。比如一个苹果，它可以是红色的也可以是绿色的。如果简单地使用 L2 损失，网络为了最小化误差，会倾向于输出这两种颜色的平均值（比如灰色或褐色），导致上色结果模糊。
**改进方案**：将 ab 通道量化为 $Q$ 个色块（如 $Q=313$），将颜色预测转化为**分类问题**，使用交叉熵损失代替 L2 损失。这样，网络可以自信地预测某一个具体的颜色，从而产生更鲜艳、更符合语义的上色效果。

**Split-Brain Autoencoder**

Split-Brain Autoencoder 将上述着色思想推广到更一般的跨通道预测中。它的核心思想是：将输入图像 $X$ 的通道切分为两组 $X_1$ 和 $X_2$，让两个网络分别用一组通道去预测另一组。

![SBA](../img/SBA.png)

其操作步骤如下：
1. **通道切分（Input Split）**

   给定输入图像 $X$，将其通道或空间结构切分为两组，记为 $X_1$（图中蓝色对角部分）和 $X_2$（图中黄色对角部分）。

   在实际实现中，有两种常见的切分方式：一是通道切分，比如在 LAB 颜色空间中，$X_1$ 是亮度通道 L，$X_2$ 是颜色通道 a 和 b；

   二是空间切分，比如 $X_1$ 是图像的上半部分，$X_2$ 是图像的下半部分。
2. **双向预测（Cross Prediction）**

   将 $X_1$ 输入到网络 $\mathcal{F}_1$ 中，得到预测输出 $\widehat{X}_2$；

   同时，将 $X_2$ 输入到网络 $\mathcal{F}_2$ 中，得到预测输出 $\widehat{X}_1$。图中虚线框内上下两条支路并行计算，正是体现了这种对称的互预测机制。

   这里的网络 $\mathcal{F}_1$ 和 $\mathcal{F}_2$ 通常是编码器-解码器结构。编码器负责将输入压缩成低维特征向量，解码器负责从特征向量中恢复出预测的图像。两个网络可以是完全独立的，也可以共享部分编码器权重。
3. **重建与损失计算（Reconstruction & Loss）**

   将预测出的 $\widehat{X}_1$ 和 $\widehat{X}_2$ 拼接组合，得到右侧完整的重建图像 $\widehat{X}$。然后计算重建图像与原始图像之间的差异，更新网络参数。

**公式原理**

这个任务使用重建误差作为损失函数，通常为均方误差（MSE），公式如下：

$$ \mathcal{L} = \mathbb{E}_{X}\left[ \| \mathcal{F}_2(X_2) - X_1 \|_2^2 + \| \mathcal{F}_1(X_1) - X_2 \|_2^2 \right] $$

其中 $\mathcal{F}_1$ 和 $\mathcal{F}_2$ 是两个网络的映射函数。损失函数由两部分对称组成：一部分是预测 $\widehat{X}_2$ 与真实的 $X_2$ 的误差，另一部分是预测 $\widehat{X}_1$ 与真实的 $X_1$ 的误差。

这个框架的底层逻辑在于：$X_1$ 和 $X_2$ 是从同一张图像中分离出来的，它们之间存在着天然的内在关联（互信息）。以图像着色为例，亮度通道 L 和颜色通道 ab 是高度相关的，边缘和纹理在 L 和 ab 上往往同时存在。网络 $\mathcal{F}_1$ 要能用亮度预测出颜色，就必须从亮度中提取出物体的形状、轮廓和语义特征，而不是简单的像素记忆。这迫使编码器学习到数据内在的深层表示，从而为下游任务提供强大的迁移能力。

**视频着色（Video Coloring）**
图像着色的逻辑可以自然地延伸到时序视频中。如果能利用视频的前一帧（参考帧）给后续帧上色，模型就不仅能理解单张图的语义，还能隐式地学会如何追踪物体在帧间的运动变化。这本质上引出了对视频中每像素邻域做 CNN 处理的注意力机制。



#### 掩码自编码器 Masked Autoencoders (MAE)

MAE 是当前最常用的基于重建的自监督学习方法之一。它的核心设计理念是**非对称的自编码器（Asymmetrical Autoencoder）**。通过极高的掩码率和极其轻量的解码器，MAE 在预训练效率和下游任务迁移上展现了极强的能力。

**网络结构与范式**

传统的自编码器（如 U-Net）通常是对称的，编码器和解码器的参数量、深度相当，输入和输出保持相同的序列长度。MAE 打破了这种对称性，将绝大部分计算负担放在了编码器上，解码器则极其轻量。

![](../img/MAE_.png)
整体数据流动可以概括为以下五个步骤：
1. **切块**：将图像划分为 $N$ 个不重叠的图像块（如 16×16）。
2. **高比例掩码**：按 75% 的比例随机遮蔽，只保留 25% 的可见块。
3. **编码**：编码器（大模型）只处理这 25% 的可见块，提取特征。
4. **解码**：解码器（小模型）接收编码器输出，并在被遮蔽的位置插入共享的可学习 Mask Tokens，使序列恢复到完整的 $N$ 个 token。
5. **重建与丢弃**：进行 Transformer 计算和线性投影，重建完整图像。训练完成后，解码器被丢弃，只保留编码器用于下游任务。

**核心机制与操作步骤**

1. **图像分块与高比例掩码（Masking）**
   - **操作**：与原始 ViT 一样，将输入图像划分为不重叠的图像块，然后均匀采样一个非常大比例（如75%）的块并将它们遮挡，只保留剩余的一小部分（25%）可见块。
   - **动机**：如果掩码率太低，网络可以仅仅依靠附近可见像素的简单插值来完成重建，这会导致模型学到的是局部平滑特征，而非高层语义。高掩码率迫使模型必须理解整个图像的全局结构和物体语义，才能补全大面积缺失的内容，这让任务变得困难且更有意义。

2. **编码器（Encoder）仅处理可见块**
   - **输入**：编码器**只对未被遮挡的 25% 图像块进行操作**。首先对可见块进行线性投影，并加上位置编码，然后输入一系列的 Transformer 块。
   - **计算特性**：由于编码器只看到 25% 的样本，它的输入序列长度很短，计算量大幅减少。因此，**MAE 的编码器可以设计得非常大**。由于输入序列短，编码器在单个 token 上的计算量甚至可以比解码器多出 9 倍以上。这使得模型可以在有限算力下参数量飙升至十亿级别。
   - **关键设计**：编码器内部**不包含 Mask Token**。如果在编码器输入时就加入 Mask Token，大量的 Mask Token 会参与 Transformer 的注意力计算，极大地增加编码器的计算量，违背了“只处理部分可见块”的初衷。MAE 选择在编码器之后才插入 Mask Token。

3. **解码器（Decoder）重建缺失像素**
   解码器专门为重建像素而设计，结构非常轻量（通常只有几层 Transformer 块，宽度较窄）。
   - **输入与拼接**：将编码器输出的可见块特征，与**共享的可学习 mask tokens** 在**之前被遮蔽的位置**进行合并，并为它们加上位置编码，构成完整的序列。
   - **处理与重建**：经过 Transformer 块处理，最后接一个线性投影，完成像素级别的重建（finalizing pixel reconstruction）。
   - **非对称与独立性**：解码器**仅负责重建，训练完成后就会被丢弃**（not used post-training）。它的设计完全独立于编码器，非常灵活，不同于传统的自编码器或 U-Net。这也正是“非对称自编码器”设计的精髓。

4. **损失函数（Reconstruction）**
   - **目标**：损失函数使用输入图像与重建图像在像素空间中的**均方误差（MSE）**。
   - **计算范围**：但在计算时，**损失仅对被遮蔽的块（masked patches）计算**。因为可见块是直接输入给模型的，对其进行重建预测意义不大，把计算资源聚焦在真正丢失的信息上，能更高效地学习特征。

**为什么是非对称的？**

这种非对称的设计是为了在有限的算力下，最大化模型的容量和表征学习能力。如果使用对称的自编码器，编码器必须处理全部 $N$ 个 token，而解码器也要处理全部 $N$ 个 token，总计算量是巨大的。MAE 通过将编码器的输入压缩到 $0.25N$，将 75% 的计算负担（Mask Token 的处理）转移给了极其轻量的解码器。这使得我们可以用一个巨大的编码器来学习特征，而不用担心训练算力爆炸。

**评估方式：线性探测 vs. 微调**

MAE 预训练完成后，需要评估其特征的质量，通常采用两种方式：
- **线性探测（Linear Probing）**：预训练模型被冻结（参数固定），仅在其后添加一个线性层来预测标签。这**用于评估预训练特征提取器表示的质量**，反映的是特征在受限条件下的线性可分性。
- **微调（Full Fine-tuning）**：预训练模型进一步训练（参数不冻结），并在后接一层或多层，可能包含非线性。这**利用模型近乎真实的潜力来适应新任务**，通常在下游任务上能达到最高精度。

**消融研究（Ablation Studies）中的关键发现**

MAE 论文进行了大量的消融实验来验证设计选择，主要结论如下：
- **掩码率（Masking Ratio）**：微调（fine-tuning）在 75% 的掩码率下达到最佳效果，而线性探测（linear probing）在 60%-70% 时达到峰值。这证明高掩码率对提取优质语义特征至关重要。
- **掩码采样方法**：对比随机采样（random）、块采样（block）和网格采样（grid）。**随机采样在 75% 掩码率下表现最好**（Random 75% 微调 84.9，线性探测 73.5）。块采样（Block 50%）和网格采样（Grid 75%）的准确率均低于随机采样。视觉上，随机 75% 的遮挡最零散，防止了模型利用局部连续性作弊，迫使它进行全局推理。
- **其他消融因素**：解码器的深度和宽度、是否在编码器中使用 mask token（MAE 是不用的，以节省计算）、重建目标（如归一化像素）、数据增强以及训练计划等，都会对结果产生影响。

#### 对比学习 Contrastive Learning

与 MAE 这种基于重建（预测像素）的方法不同，对比学习是判别式自监督学习的主流代表。它的核心思想是：**在特征空间中，把来自同一个对象的样本拉近，把不属于同一个对象的样本推远。** 它不关心图像的具体像素细节，只关心不同样本之间的相对距离。

**核心概念：正负样本与评分函数**

在对比学习中，监督信号是通过构造正负样本对自动生成的。
- **正样本对**：通常是对同一张图片施加不同的数据增强（如裁剪、颜色抖动）得到的两个视图。它们底层语义相同，在特征空间里应该靠近。
- **负样本对**：来自不同图片的视图。它们在特征空间里应该远离。
- **评分函数（Score Function）**：通常使用**余弦相似度（Cosine Similarity）**来衡量两个特征向量的相似程度。给定两个向量 $u$ 和 $v$，相似度函数记为 $s(u, v)$。

**公式化定义：InfoNCE 损失与互信息**

给定 1 个正样本和 $N-1$ 个负样本，对比学习的损失函数被严格定义为：
$$ L = -\mathbb{E}_X \left[ \log \frac{\exp(s(f(x), f(x^+)))}{\exp(s(f(x), f(x^+))) + \sum_{j=1}^{N-1} \exp(s(f(x), f(x_j^-)))} \right] $$
其中 $f(\cdot)$ 是编码器，$x^+$ 是正样本，$x_j^-$ 是负样本。这个损失函数被称为 **InfoNCE loss**。

它本质上是一个**多分类交叉熵**问题，目的是让模型在 $N$ 个候选中（1个正样本 + $N-1$个负样本）将正样本分类出来。实际应用中，通常会在相似度分数上除以温度参数 $\tau$（即 $s(u,v)/\tau$）来调节 softmax 的平滑程度。

**为什么叫 InfoNCE？** 因为它隐式地最大化了编码器输出 $f(x)$ 和 $f(x^+)$ 之间的**互信息（Mutual Information, MI）**。具体来说，InfoNCE 损失 $L$ 是互信息的一个**下界（lower bound）**：
$$ MI[f(x), f(x^+)] - \log(N) \ge -L $$
**负样本大小 $N$ 越大，这个下界就越紧（tighter bound）**，互信息估计就越准确。这一结论完美解释了为什么 SimCLR 需要极大的 batch size，而 MoCo 需要极大的负样本队列——因为 $N$ 越大，模型学到的特征表示质量就越高。

**SimCLR**

SimCLR 是对比学习中非常经典且直观的一个框架。

**操作步骤：**
1. 取一张无标签图片，对它施加两次不同的随机数据增强，得到两个视图 $x_i$ 和 $x_j$。
2. 将这两个视图分别送入**同一个编码器**提取特征。
3. 将特征通过一个**非线性投影头**（MLP）映射到对比空间，得到 $z_i$ 和 $z_j$。
4. 在一个 batch 中，将 $(z_i, z_j)$ 作为正样本对，其他 $2(N-1)$ 个特征作为负样本。
5. 计算 InfoNCE 损失，并通过反向传播更新编码器和投影头。


**mini-bacth training**

![mini-b t](../img/miniSimCLR.png)

SimCLR 在一个 mini-batch 中的具体训练过程如下：

1. **构造正样本对与编码**：在一个 batch 中取 $N$ 张图片，对每张图片做两次不同的数据增强，得到 $2N$ 张增强图（如图中上下两排）。它们经过共享参数的编码器（encoder）后，输出特征矩阵 $z \in \mathbb{R}^{2N \times D}$。其中第 $2k$ 和第 $2k+1$ 个元素构成一个正样本对。
2. **计算亲和力矩阵（Affinity matrix）**：对于这 $2N$ 个特征向量，计算两两之间的余弦相似度，得到 $2N \times 2N$ 的相似度矩阵。公式为：
   $$ s_{i,j} = \frac{z_i^T z_j}{\|z_i\| \|z_j\|} $$
3. **转化为多分类问题**：在这个矩阵中，**每一行代表一个样本**。对于第 $i$ 行，它的正样本（即另一个视图）所在的位置就是它的“分类标签”（图中标注的 "classification label for each row"）。同一行的其他 $2N-2$ 个位置全是负样本。
4. **计算损失**：由于每一行都有唯一的正样本标签，SimCLR 直接对矩阵的每一行使用 **InfoNCE 损失（即交叉熵）**，迫使模型在 $2N-1$ 个候选中正确分类出正样本。





**关键发现与局限：**
非线性投影头和强数据增强对于 SimCLR 的成功至关重要，它们迫使编码器学习到对变换鲁棒的高层语义特征。但它的局限在于性能高度依赖于**非常大的 batch size**。因为负样本直接从当前 batch 中获取，batch 越大，$N$ 越大，下界越紧，对比学习的效果越好，但这也极大消耗了显存。

**MoCo (Momentum Contrast)**

MoCo 针对 SimCLR 依赖大 batch 的痛点进行了改进，核心在于**解耦了 batch size 与负样本数量**，使得在有限显存下也能获得极大的 $N$。

**核心机制：**
- **负样本队列（Queue）**：MoCo 维护了一个固定大小的队列，用来存储历史 batch 计算出的 key 特征。队列中的特征作为负样本参与对比学习。即使 batch size 很小（如 256），队列里也能有成千上万个负样本（$N$ 很大），从而让 InfoNCE 的下界更紧。
- **动量编码器（Momentum Encoder）**：MoCo 有两个编码器：Query 编码器和 Key 编码器。Query 编码器通过梯度正常更新，而 Key 编码器不参与反向传播，而是通过**动量更新**来缓慢跟进 Query 编码器的参数：
  $$ \theta_k \leftarrow m \theta_k + (1 - m) \theta_q $$

**为什么不能对队列里的 Key 进行反向传播？**
因为队列中的特征来自之前的多个时间步，如果对它们进行反向传播，计算图会极其庞大，显存无法承受。更重要的是，历史队列中特征的参数已经更新过，直接回传梯度会导致训练极其不稳定。因此，MoCo 选择**停止梯度回传**，用动量更新的方式让 Key 编码器“温和”地跟上 Query 编码器的步伐，保证了队列中特征的一致性。


DINO
DINOv2 （更大的训练数据）能生成很强的自监督学习特征




#### 生成模型

监督学习和无监督学习

x到y的映射（监督学习）
聚类或者降维PCA（无标签，无监督学习），寻找隐藏的结构

生成模型与判别模型

同一张图上不同标签进行概率竞争。判别模型无法拒绝不合理的输入

无条件生成模型
所有图像都在争夺概率状态。
生成一个覆盖x的概率分布
可以拒绝
（实际意义不大）剔除异常值

条件生成模型

所有可能的标签产生竞争。
产生新数据

贝叶斯法则将上述三个模型联系起来（理论上能用另外的两个模型推出另一个条件模型，不过一般不这么做）


为什么关注生成式建模

输出存在歧义


语言建模



自回归模型Autoregressive Models

最大似然估计（通用核心思路）

用RNN或者masked Transformer

假设存在某种规范方式，能把数据x拆分开来（把一个样本拆分开来）
需要把数据拆成序列（对文本是比较自然的）（直接用来处理图像，会数据太长（拉成一维像素序列））


VAEs Variational Autoencoders变分自编码器

近似某种密度，
学习过程会自然产生代表数据的向量


非变分自编码器


无监督学习

输入x提取特性向量z（包含输入x的有用信息）

假设有一个模型模仿恒等函数

加入瓶颈z迫使模型学习数据的非平凡结构（挤过瓶颈层）

通过解码器，由z生成新的图像。

如果能抛弃解码器，采样生成一个new z ，我们可以生成新的图像


如何生成new z（这并不简单）

VAE（近似密度p(x)）
假设存在假设空间z，z向量包含了图像的所有信息
强制自编码器具有概率性，给潜在空间施加概率结构
强制z服从已知分布（这样就具有z的样本了）
假设一个简单的先验，先验几乎都用高斯分布

假装知道z

但没法对z积分，因此试着使用贝叶斯公式

创建一个新的神经网络近似P（z|x）

联合训练编码器Q 和解码器P去最大化变分下界ELBO


**DINO 与 DINOv2**

DINO（Self-Distillation with No Labels）采用自蒸馏机制，通过动量编码器让学生网络去预测教师网络的输出。DINOv2 将其进一步扩展到了更大的训练数据集上，学习到了非常强大的通用视觉特征，是目前最常用的自监督预训练模型之一。

---

#### 生成模型 Generative Models

**监督学习与无监督学习**

监督学习本质上是在寻找从输入 $x$ 到标签 $y$ 的映射。而无监督学习（如聚类、PCA降维）在没有标签的情况下，试图寻找数据中隐藏的结构。

**生成模型与判别模型**的核心差异在于：判别模型只能对同一张图的不同标签进行概率竞争，它无法拒绝不合理的输入；而生成模型（尤其是无条件生成模型）是在所有图像之间争夺概率状态，它生成一个覆盖 $x$ 的概率分布，因此它具备**拒绝异常值**的能力。

条件生成模型则是在所有可能的标签之间产生竞争，并据此生成新数据。

理论上，通过贝叶斯法则可以将上述三个模型联系起来，但实际中一般不这么做。

为什么我们需要关注生成式建模？一个重要原因是**输出存在歧义**。例如预测视频的下一帧或进行语言建模，输入往往对应多种合理的输出，生成模型能更好地处理这种概率分布。

**生成模型的分类 Taxonomy**

![GM](../img/GM.png)

生成模型分为两大类，核心区别在于它们如何对概率分布 $P(x)$ 进行建模：

- **显式密度模型（Explicit Density）**：显式地计算或近似 $P(x)$。
  - **可计算密度（Tractable Density）**：可以直接计算出 $P(x)$ 的值，如自回归模型（Autoregressive）。
  - **近似密度（Approximate Density）**：无法直接计算，需要用变分法近似，如变分自编码器（VAE）。
- **隐式密度模型（Implicit Density）**：不显式计算 $P(x)$，但能从分布中采样生成数据。
  - **直接采样（Direct）**：如生成对抗网络（GAN）。
  - **间接采样（Indirect）**：如扩散模型（Diffusion Models）。


#### 自回归模型 Autoregressive Models

**显式密度建模目标**

自回归模型是**显式密度模型（Explicit Density）**中最直接的一类。它的核心目标是写出一个显式的函数 $p(x) = f(x, W)$，用来直接计算数据 $x$ 的概率分布。这与 GAN 这种隐式模型（只采样不计算概率）和 VAE 这种近似模型（计算下界）有本质区别。

**序列假设与链式法则**

为了写出这个概率函数，模型假设数据 $x$ 可以被拆解为一个序列：$x = (x_1, x_2, ..., x_T)$。接着，利用**概率的链式法则（Chain Rule of Probability）**，将联合概率分解为条件概率的乘积：

$$ p(x) = p(x_1, x_2, ..., x_T) $$
$$ = p(x_1)p(x_2 | x_1)p(x_3 | x_1, x_2) ... $$
$$ = \prod_{t=1}^T p(x_t | x_1, ..., x_{t-1}) $$

这个公式的核心含义是：**每个元素的概率，都依赖于它之前的所有元素。** 前提是已知序列此前的所有部分，模型才能预测当前时刻的下一个元素。

**架构：以 RNN 语言建模为例**

![AM](../img/AM.png)

图片左侧展示了使用 RNN 进行语言建模的展开图。数据流完全契合链式法则：
- 初始输入 $x_0$（通常为起始符）进入第一个隐藏状态 $h_1$，输出第一个元素的条件概率 $p(x_1)$。
- 接着，真实的前一时刻输入 $x_1$ 和隐藏状态 $h_1$ 共同传入下一步，得到 $h_2$，输出 $p(x_2 | x_1)$。
- 以此类推，每一步 $h_t$ 都综合了之前所有的历史信息，计算出 $p(x_t | x_1, ..., x_{t-1})$。

除了 RNN，现代自回归模型更多使用 **Masked Transformer**（如 GPT 系列）来替代 RNN，并行处理更长的序列，但核心的“预测下一个 token”的逻辑是完全一致的。

**自回归模型在图像上的局限**

虽然自回归模型在文本生成上非常成功，但直接用于图像却面临巨大挑战。如果将图像拉成一维像素序列（如 PixelRNN），序列长度会极其庞大（例如 256x256 的图像有 65536 个像素），计算成本高到难以承受。此外，逐个像素生成本质上缺乏全局规划，模型容易生成局部合理但整体结构崩坏的图像。



#### 变分自编码器 Variational Autoencoders(VAE)

VAE 是近似密度模型的代表。传统的非变分自编码器（AE）通过输入 $x$ 提取特征向量 $z$，然后通过解码器由 $z$ 重建图像。

为了让模型学习到数据的内在结构，通常会在中间加入一个**瓶颈(Bottleneck)**层。

但这面临一个问题：**如何生成新的 $z$？** 直接从 AE 的隐空间随机采样往往无法生成合理的图像。

VAE 的核心改进是给潜在空间施加概率结构。它假设训练数据 $\{x^{(i)}\}_{i=1}^N$ 是由某个不可观测的隐变量 $z$ 生成的。生成过程是：先从先验分布 $p_{\theta^*}(z)$ 中采样 $z$，再通过条件分布 $p_{\theta^*}(x|z)$ 生成 $x$。我们通常假设先验分布是简单的标准高斯分布。

**从贝叶斯法则到变分后验**

为了生成数据，VAE 的生成过程是：从先验分布 $p_{\theta^*}(z)$ 中采样 $z$，再通过条件分布 $p_{\theta^*}(x|z)$ 生成 $x$。

训练时，最基础的思路是**最大似然估计**。根据贝叶斯法则：
$$ p_\theta(x) = \frac{p_\theta(x|z)p_\theta(z)}{p_\theta(z|x)} $$
但这里存在一个问题：**$p_\theta(z|x)$ 无法计算**（分母不可积）。
**解决方案**：引入一个新的神经网络，用 $q_\phi(z|x)$ 去近似 $p_\theta(z|x)$，这就是变分后验（Variational Posterior）。

**编码器与解码器的结构**

- **编码器（Encoder）**：输入数据 $x$，输出隐变量的分布。假设该分布为高斯分布 $q_\phi(z|x) = \mathcal{N}(\mu_{z|x}, \Sigma_{z|x})$。网络输出均值和方差两个向量。
- **解码器（Decoder）**：输入隐变量 $z$，输出原始数据 $x$ 的分布 $p_\theta(x|z) = \mathcal{N}(\mu_{x|z}, \sigma^2)$。网络输出预测的数据均值 $\mu_{x|z}$。
- 如果固定方差 $\sigma^2$，最大化 $\log p_\theta(x|z)$ 等价于最小化 $x$ 与网络输出 $\mu_{x|z}$ 之间的 L2 距离（即重建误差）。

**ELBO 推导（变分下界）**

我们需要最大化边缘似然 $\log p_\theta(x)$，但直接计算不可行。推导过程如下：

1. 引入变分后验 $q_\phi(z|x)$，乘一个形为 1 的项：
   $$ \log p_\theta(x) = \log \frac{p_\theta(x|z)p(z)}{p_\theta(z|x)} = \log \frac{p_\theta(x|z)p(z)q_\phi(z|x)}{p_\theta(z|x)q_\phi(z|x)} $$

2. 拆开对数：
   $$ = \log p_\theta(x|z) - \log \frac{q_\phi(z|x)}{p(z)} + \log \frac{q_\phi(z|x)}{p_\theta(z|x)} $$

3. 对 $z \sim q_\phi(z|x)$ 取期望（期望不依赖于 $z$ 的常数项可提出）：
   $$ = E_z[\log p_\theta(x|z)] - E_z\left[\log \frac{q_\phi(z|x)}{p(z)}\right] + E_z\left[\log \frac{q_\phi(z|x)}{p_\theta(z|x)}\right] $$

4. 将期望转化为 KL 散度形式：
   $$ = E_{z \sim q_\phi(z|x)}[\log p_\theta(x|z)] - D_{KL}(q_\phi(z|x) \| p(z)) + D_{KL}(q_\phi(z|x) \| p_\theta(z|x)) $$

5. 因为 KL 散度永远大于等于 0，所以最后一项 $D_{KL}(q_\phi(z|x) \| p_\theta(z|x)) \ge 0$。我们可以直接将它丢弃，从而得到 $\log p_\theta(x)$ 的一个下界，即 **ELBO（Evidence Lower Bound）**：
   $$ \log p_\theta(x) \ge E_{z \sim q_\phi(z|x)}[\log p_\theta(x|z)] - D_{KL}(q_\phi(z|x) \| p(z)) $$
   最大化 ELBO 就等价于最大化边缘对数似然 $\log p_\theta(x)$。

**VAE 的训练过程**

![VAETrain](../img/VAETraining.png)

训练目标是最大化 ELBO。

1. **编码器提取分布**：将输入数据 $x$ 送入编码器（Encoder），得到隐变量 $z$ 的分布参数。编码器输出两个向量：均值 $\mu_{z|x}$ 和标准差 $\Sigma_{z|x}$。这构成了变分后验分布 $q_\phi(z|x) = \mathcal{N}(\mu_{z|x}, \Sigma_{z|x})$。
2. **先验损失（Prior Loss）**：为了让隐空间具有生成能力，编码器输出的分布应该接近单位高斯分布（零均值、单位方差）。这体现在 ELBO 的 KL 散度项中。它希望 $\Sigma_{z|x} = I$ 且 $\mu_{z|x} = 0$。
3. **重参数化采样（Reparameterization Trick）**：从编码器输出的分布中采样 $z$ 这个操作本身是不可导的。通过引入随机噪声 $\epsilon \sim \mathcal{N}(0, I)$，令 $z = \epsilon \odot \Sigma_{z|x} + \mu_{z|x}$，将随机性转移到 $\epsilon$ 上，使得 $z$ 对网络参数可导。
4. **解码器重建预测**：将采样得到的隐变量 $z$ 送入解码器（Decoder），得到预测的数据均值 $\mu_{x|z}$，对应分布 $p_\theta(x|z) = \mathcal{N}(\mu_{x|z}, \sigma^2)$。
5. **重建损失（Reconstruction Loss）**：预测的数据均值 $\mu_{x|z}$ 应该与原始输入 $x$ 尽可能一致。如果固定方差 $\sigma^2$，最大化 $\log p_\theta(x|z)$ 等价于最小化它们之间的 L2 距离。

**网络结构与损失函数的两项对抗**

在训练时，整个网络的优化目标是最大化变分下界（ELBO），对应于幻灯片上方绿色框出的公式：
$$ E_{z \sim q_\phi(z|x)}[\log p_\theta(x|z)] - D_{KL}(q_\phi(z|x) \| p(z)) $$

这个损失函数由两项组成，它们相互对抗：
- **重建损失（第一项）**：期望项 $E_{z \sim q_\phi(z|x)}[\log p_\theta(x|z)]$。它希望 $\Sigma_{z|x} = 0$ 且 $\mu_{z|x}$ 对每个 $x$ 都是独一无二的，这样解码器就能确定性地重建 $x$，并让损失的重建项主要作用于解码器输出。
- **先验损失（第二项）**：KL 散度项 $- D_{KL}(q_\phi(z|x) \| p(z))$。它希望 $\Sigma_{z|x} = I$ 且 $\mu_{z|x} = 0$，即强制编码器输出始终服从单位高斯分布。

这两项在训练中相互制约：重建损失想让 $z$ 尽可能保留信息，先验损失想让 $z$ 尽可能标准化。最终的损失函数在这两者之间寻找平衡。


**采样生成图像**

训练完成后，生成新图像非常简单：
1. 从先验分布中采样 $z \sim \mathcal{N}(0, I)$。
2. 将 $z$ 输入解码器，得到条件概率分布 $p_\theta(x|z) = \mathcal{N}(\mu_{x|z}, \sigma^2)$。
3. 取解码器输出的均值 $\mu_{x|z}$ 作为生成的图像像素。



#### 生成对抗网络 Generative Adversarial Networks (GANs)


GANs 属于**隐式密度模型（Implicit Density Model）**。它不尝试显式地计算真实数据分布 $p_{data}(x)$ 的具体概率值，而是通过建模，用一个神经网络来近似并逼近真实分布，最终目标是让生成器分布 $p_G$ 等于真实分布 $p_{data}$。

它的核心做法是引入一个**潜在变量 $z$**，它遵循一个已知的简单先验分布（通常为标准单位高斯分布 $p(z) = \mathcal{N}(0, I)$）。接着，将 $z$ 送入生成器网络 $G$，得到生成的图像 $x = G(z)$。这是一个从简单分布到复杂数据分布的映射。

**网络结构与工作流程**


GAN 由两个独立但联合训练的神经网络组成：
- **生成器网络（Generator Network, G）**：接收随机噪声 $z$，将其转换为假图像（Fake image）。它的目标是尽量生成逼真的图像，骗过判别器。
- **判别器网络（Discriminator Network, D）**：接收图像（真实图像或生成图像），输出一个概率值 $D(x)$，表示 $x$ 是真实图像的概率（$D(x) = 1$ 代表真，$D(x) = 0$ 代表假）。它的目标是尽可能准确地将真图和假图区分开来。

**极小极大博弈（Minimax Game）与损失函数**

![](../img/GAN_D.png)

GAN 的训练本质上是一个**极小极大博弈**。判别器 $D$ 试图最大化目标函数 $V(G, D)$，而生成器 $G$ 试图最小化它。联合训练的目标函数如下：
$$ \min_G \max_D V(G, D) = \mathbb{E}_{x \sim p_{data}}[\log D(x)] + \mathbb{E}_{z \sim p(z)}[\log (1 - D(G(z)))] $$

从这个公式可以看出：
- 判别器 $D$ 希望真实数据 $x$ 的预测 $D(x)$ 接近 1，希望生成数据 $G(z)$ 的预测 $D(G(z))$ 接近 0。
- 生成器 $G$ 希望判别器对生成数据的预测 $D(G(z))$ 接近 1。

**梯度消失问题与改进（非饱和损失）**
![GAN_Training](../img/GAN_Train_.png)

在训练初期，生成器 $G$ 的能力很弱，生成的图像质量很差，判别器 $D$ 很容易就能分辨真假，此时 $D(G(z))$ 趋近于 0。
此时如果直接使用 $\log(1 - D(G(z)))$ 作为 $G$ 的损失，由于 $D(G(z))$ 接近 0，损失函数曲线非常平缓，**导致对 $G$ 的梯度接近 0**，生成器无法得到有效更新。

解决方案是：生成器不再最小化 $\log(1 - D(G(z)))$，而是改为**最小化 $-\log(D(G(z)))$**。从右侧的曲线图可以看出，当 $D(G(z))$ 接近 0 时，$-\log(D(G(z)))$ 的曲线非常陡峭，能提供强大的梯度，让生成器在初期就能快速学习。

**交替训练与梯度更新**


训练过程采用**交替梯度更新**。在每一轮迭代中：
1. **训练判别器**：固定生成器 $G$，对 $D$ 进行梯度上升，最大化 $V$。
   $$ D = D + \alpha_D \frac{dV}{dD} $$
2. **训练生成器**：固定判别器 $D$，对 $G$ 进行梯度下降，最小化 $V$。
   $$ G = G - \alpha_G \frac{dV}{dG} $$

这里有两个重要的直觉：首先，目标函数 $V$ 的具体数值本身**没有实际意义**，它只是博弈过程中的一个度量值。其次，整个训练是一个**非平稳分布**的过程：在训练生成器时，判别器的参数也在变化；在训练判别器时，生成器的分布也在变化。因此，训练初期判别器没那么好并不关键，因为生成器也在不断进步。

**潜空间插值与平滑特征**

GAN 的一个非常优雅的特性是它能在**潜在空间里学到平滑的特征**。如果我们在潜空间中取两个随机向量 $z_0$ 和 $z_1$，对它们进行线性插值：
$$ z_t = t z_0 + (1 - t) z_1 $$
$$ x_t = G(z_t) $$
生成器 $G$ 输出的图像 $x_t$ 会在两张图之间平滑地变形过渡。这种平滑性使得 GAN 具有强大的图像编辑和合成能力。代表模型包括 DC-GAN 和 StyleGAN。

**局限与挑战**

GAN 非常强大，但它**很难训练，且看不到直观的损失曲线**（因为损失值本身没有意义，它只反映了两个网络的博弈状态）。训练过程中极易出现模式崩溃（生成器只能生成少数几种样本）或训练不稳定（振荡）等问题。因此，在后续的生成模型发展中，扩散模型（Diffusion Models）逐渐成为新的主流方向。


#### 扩散模型 Diffusion Models



扩散模型的灵感来源于热力学中的扩散现象。它的核心直觉非常直观：**先破坏，再重建。**

具体流程分为正向加噪和反向去噪两个过程：
1. **正向加噪**：给定一张真实图像 $x$，我们不断往里面加入高斯噪声。噪声等级从 $t=0$ 到 $t=1$，当 $t=0$ 时是没有噪声的原图，当 $t=1$ 时图像完全变成纯噪声。这可以看作是一个逐渐破坏图像信息的过程。
2. **反向去噪**：训练一个神经网络 $f_\theta(x_t, t)$，它的任务是**每次去除一点点噪声**。输入带有噪声的图像 $x_t$ 和当前噪声等级 $t$，网络预测出需要去除的噪声。
3. **采样生成**：在推理时，从一个纯高斯噪声 $x_1 \sim p_{noise}$ 开始，反复应用网络 $f_\theta$ 多次（序列化地去噪），最终生成一张无噪的样本 $x_0$。

这里有一点需要特别注意：**噪声分布的形状必须与数据完全一样**。这意味着如果数据是图像，噪声也是一个与图像尺寸相同的张量，而不是一个低维的向量。

#### 整流流模型 Rectified Flow

传统的扩散模型通过随机微分方程（SDE）或常微分方程（ODE）来定义加噪和去噪过程，数学上较为复杂。**整流流（Rectified Flow）** 提出了一种更简单、更直观的替代方案：**直接学习从噪声分布到数据分布的直线轨迹。**

**公式原理**

假设我们有简单的噪声分布 $p_{noise}$（如标准高斯）和真实数据分布 $p_{data}$。在每次训练迭代中，我们分别采样 $z \sim p_{noise}$、$x \sim p_{data}$ 以及时间步 $t \sim \text{Uniform}[0, 1]$。

我们将噪声和数据按时间步进行线性插值，得到中间带噪样本 $x_t$：
$$ x_t = (1-t)x + tz $$
同时，从数据指向噪声的**速度向量 $v$** 被定义为：
$$ v = z - x $$
模型的目标就是预测这个速度向量。损失函数为均方误差：
$$ L = \| f_\theta(x_t, t) - v \|_2^2 $$

![RF](../img/RF.png)

**实现**

整流流的实现极其精简。模型的核心训练循环只有几行代码：
```python
# 核心训练循环
for x in dataset:
    z = torch.randn_like(x)
    t = random.uniform(0, 1)
    xt = (1 - t) * x + t * z
    v = model(xt, t)  # 模型预测速度
    loss = (z - x - v).square().sum()
```

在采样时，我们从纯噪声 $x \sim p_{noise}$ 开始，选择一个步数 $T$（通常 $T=50$），然后以欧拉法逐步迭代：
$$ x = x - v_t / T $$
这里 $v_t = f_\theta(x_t, t)$ 是模型预测的速度。因为速度场是直线，通常可以在非常少的步数内（如 50 步甚至更少）完成高质量采样，比传统扩散模型快得多。

#### 条件生成与 Classifier-Free Guidance (CFG)

**条件信息的注入**

要生成特定类别（如“狗”或“猫”）的图像，就必须引入条件信息 $y$。在条件整流流（Conditional Rectified Flow）中，训练时模型接收 $x_t, y, t$，预测速度 $v$。采样时，用户输入条件 $y$，模型在每一轮迭代中都会根据这个 $y$ 来调整生成方向：
```python
# 条件采样
y = user_input()
sample = torch.randn(x_shape)
for t in torch.linspace(1, 0, num_steps):
    v = model(sample, y, t)
    sample = sample - v / num_steps
```

**CFG 的原理与代价**

![CFG](../img/CFG.png)

在训练条件模型时，如果我们总是强制性地提供条件 $y$，模型可能会过度依赖它。为了让模型既能生成有条件的图像，也能生成无条件的图像，**Classifier-Free Guidance (CFG)** 提出了一个巧妙的训练策略：在训练阶段，**随机以一定概率（如 0.5）将条件 $y$ 替换为 null（记作 $y_{null}$）**。

在采样阶段，CFG 同时计算两次预测：一次带条件 $v_y = f_\theta(x_t, y, t)$，一次无条件 $v_0 = f_\theta(x_t, y_{null}, t)$。然后将它们组合：
$$ v = (1+w)v_y - wv_0 $$
其中 $w$ 是引导强度。这样做的目的是放大条件对生成结果的影响，让生成的图像更符合文本或标签的要求。

CFG 在实践中非常重要，能显著提升生成质量，但代价是**采样成本翻倍**（每次迭代需要跑两次模型）。

#### 扩散 Transformer (Diffusion Transformer, DiT)

传统的扩散模型通常使用 U-Net 作为骨干网络，但**DiT 将 U-Net 替换为了标准的 Transformer 块**。

**如何注入条件信息**

DiT 的核心问题在于：如何将时间步 $t$、文本 $y$ 等条件信息注入到 Transformer 中？结合架构图，主要有以下几种机制，不同模型往往两者都用：
1. **预测缩放/偏移（Predict scale/shift）**：最常见的方式。将时间步 $t$ 通过 MLP 映射，输出一组缩放因子 $\alpha$ 和偏移量 $\beta$，直接作用于 Transformer 的层归一化（LayerNorm）特征上，类似于 AdaIN 的思想。
2. **交叉注意力（Cross-Attention）**：将条件信息作为 Key 和 Value，将图像隐变量作为 Query。这种方式非常灵活，是文本到图像生成的主流做法。
3. **联合注意力（Joint Attention）**：将条件序列（如文本 Token）与图像序列拼接在一起，共同送入多头自注意力机制中，让模型在生成图像时能全局地关注到所有的条件信息。

至此，扩散模型的基础框架、训练方式、条件生成和架构改进已经完整。接下来，为了将其扩展到高分辨率图像，我们需要进入下一部分：Latent Diffusion Models (LDMs)。


#### 潜在扩散模型 Latent Diffusion Models (LDMs)

**动机：为什么需要 LDMs**

传统的扩散模型直接在**像素空间（Pixel Space）**上进行加噪和去噪。虽然生成质量极高，但它有两个致命的缺陷：一是计算开销巨大，处理高分辨率图像时，反向去噪的每一步都需要在整个高清像素矩阵上进行；二是很难扩展到高分辨率（如 1024x1024），因为时间和显存消耗会随着像素规模呈平方级增长。

LDMs 的核心解决方案是：**将扩散过程转移到压缩的潜在空间（Latent Space）中进行。** 这样可以大幅降低计算量，同时保留生成高质量图像的能力。

**两阶段训练机制**

LDMs 的训练分为两个阶段，两者相互独立：

第一阶段，训练一个**自编码器（Autoencoder）**，包括编码器（Encoder）和解码器（Decoder）。编码器将原始图像 $x$ 压缩成低维的潜在表示 $z$，解码器负责从 $z$ 重建回图像。这一步大幅降低了空间维度（例如 $H \times W \times 3$ 压缩为 $H/D \times W/D \times C$，$D$ 通常是 8 或 16）。
第二阶段，**冻结编码器**，然后训练一个**扩散模型**。这个扩散模型不再处理原始图像，而是专门去学习如何去除加在潜在表示 $z$ 上的噪声。训练完成后，采样过程是：从纯噪声潜变量开始，逐步去噪得到干净的潜变量，最后送入解码器，得到最终图像。

**解决解码器模糊：VAE + GAN + Diffusion**

![LDMs](../img/LDMs.png)

单纯使用 VAE 训练的自编码器存在一个问题：**解码器输出的图像往往比较模糊**。因为纯 MSE 重建损失会让模型倾向于输出像素平均值，缺乏高频细节。

解决方案是**引入判别器（Discriminator）**。具体来说，LDM 的第一阶段实际上是在训练一个类似 VAE-GAN 的架构。除了常规的 VAE 重建损失和 KL 散度（通常 KL 权重设得非常小），还加入了一个判别器网络，让它去区分“真实图像”和“VAE 解码器输出的假图像”。生成器（即编码器+解码器）则要想办法骗过判别器。这种对抗训练机制（VAE + GAN + Diffusion 的组合）使得解码器能够重建出更清晰、更锐利的图像细节。

#### 扩散 Transformer Diffusion Transformer (DiT)

**核心架构与条件注入**

早期扩散模型通常使用 U-Net 作为骨干网络，而 **DiT（Diffusion Transformer）** 将 U-Net 替换为标准的 Transformer 块，证明了 Transformer 在扩散模型中同样极其强大（后续的 Sora 等视频生成模型正是基于此架构）。

DiT 的核心问题在于：**如何将条件信息（含噪图像、时间戳信息、文本等）注入到 Transformer 中？** 主要有以下几种机制：

1. **预测缩放/偏移（Predict scale/shift）**：这是最常见的方式，主要用于注入**时间戳信息**。将时间步 $t$ 通过 MLP 映射，输出一组缩放因子 $\alpha$ 和偏移量 $\beta$，直接作用于 Transformer 的层归一化（LayerNorm）特征上，类似于 AdaIN 的思想。
2. **交叉注意力（Cross-Attention）**：这是注入**文本或类别条件**的常见方式。将条件信息作为 Key 和 Value，将图像隐变量作为 Query。这种方式非常灵活。
3. **联合注意力（Joint Attention）**：将条件序列（如文本 Token）与图像序列拼接在一起，共同送入多头自注意力机制中，让模型在生成图像时能全局地关注到所有的条件信息。

#### 扩散模型的采样与加速

**采样缓慢的瓶颈**

扩散模型在推理（采样）时非常慢。因为训练时是逐步加噪，采样时就必须**从纯噪声开始，逐步反向去噪**。结合幻灯片，通常需要几十步（如 T=50）甚至上千步的迭代。每一步都需要把整个模型跑一遍，如果再加上 CFG 的双倍计算，时间开销极其巨大。

**蒸馏算法（Distillation）**

为了加速采样，**蒸馏算法**成为了关键的研究方向。其核心思想是：让一个**学生模型（Student Model）**去学习**教师模型（Teacher Model，即完整的扩散模型）**在多步采样中的行为，目标是在少数推理步数（如 4 步甚至 1 步）内，尽可能保留教师模型的生成质量。蒸馏算法的关键在于**在极少步数下保存质量**，这是目前扩散模型落地应用必须攻克的工程难题。

**Logit-normal 采样**

除了蒸馏，调整训练和采样时的时间步采样策略也能提升效果。**Logit-normal 采样**是一种在训练时用于选择时间步 $t$ 的分布策略。相比于均匀采样，logit-normal 分布会让模型更多地关注某些特定区间的时间步（例如中间噪声水平的阶段），从而更有效地学习去噪过程，最终提升生成质量。